# **Laboratory of Wearable Biosignal Processing**

## **03 - Neural Networks for Biosignals: Preprocessing and Training Pipeline**

## Before You Start

```bash
git pull
```


## Dataset: PTB-XL (Subset)

Same dataset as previous sessions: subset of the [PTB-XL](https://physionet.org/content/ptb-xl/1.0.3/) dataset (first 500 subjects, 12-lead ECG at 100/500 Hz).


## Neural Network Setup (PyTorch + CUDA Check)

Before training neural networks, we verify whether a CUDA-capable GPU is available and select the compute device accordingly.

In [ ]:
import torch

import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader, TensorDataset

# Select compute device for PyTorch models.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Selected device: {device}")

if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"CUDA runtime version: {torch.version.cuda}")
else:
    print("Running on CPU. If you expect GPU acceleration, verify CUDA drivers and PyTorch CUDA build.")

## Data Loading and Quick Visualization

Load all ECG records and plot a sample 12-lead record as a sanity check.

In [ ]:
# Imports for data loading, handling, plotting, filtering, ECG peak analysis, and neural networks
import ast
import os
import wfdb

import matplotlib.pyplot as plt
import neurokit2 as nk
import numpy as np
import pandas as pd
import seaborn as sns

from scipy import signal
from sklearn.metrics import (
    accuracy_score,
    auc,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split

In [ ]:
# Root folder containing PTB-XL metadata and waveform files.
DATA_DIR = "../data"
# Fixed seed to keep random record selection reproducible across runs.
SEED = 42

# Output folder for all generated figures (grouped by notebook context).
NOTEBOOK_STEM = "03_NN_for_biosignals_pipeline"
NOTEBOOK_CONTEXT = os.path.basename(os.getcwd())

OUTPUTS_DIR = os.path.abspath(os.path.join("..", "outputs", f"{NOTEBOOK_CONTEXT}_{NOTEBOOK_STEM}"))
if not os.path.exists(OUTPUTS_DIR):
    os.makedirs(OUTPUTS_DIR, exist_ok=True)

# Folder for persisted trained models/checkpoints.
MODELS_DIR = os.path.abspath(os.path.join("..", "models"))
if not os.path.exists(MODELS_DIR):
    os.makedirs(MODELS_DIR, exist_ok=True)

# Global counter to enforce ordered names: figure_01_..., figure_02_..., etc.
FIGURE_COUNTER = 1


In [ ]:
def save_figure(fig, suffix):
    """Save a matplotlib figure using the required sequential naming convention."""
    global FIGURE_COUNTER
    filename = f"figure_{FIGURE_COUNTER:02d}_{suffix}.png"
    output_path = os.path.join(OUTPUTS_DIR, filename)
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    print(f"Saved figure: {output_path}")
    FIGURE_COUNTER += 1

def save_model_checkpoint(model, filename, metadata=None):
    """Save a trained model checkpoint in the models folder."""
    checkpoint_path = os.path.join(MODELS_DIR, filename)
    payload = {
        "model_state_dict": model.state_dict(),
    }
    if metadata is not None:
        payload["metadata"] = metadata

    torch.save(payload, checkpoint_path)
    print(f"Saved model checkpoint: {checkpoint_path}")
    return checkpoint_path

def load_model_checkpoint(model, filename, device):
    """Load model weights from the models folder and return optional metadata."""
    checkpoint_path = os.path.join(MODELS_DIR, filename)
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

    checkpoint = torch.load(checkpoint_path, map_location=device)
    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])
        metadata = checkpoint.get("metadata", None)
    else:
        # Backward compatibility: direct state_dict checkpoint.
        model.load_state_dict(checkpoint)
        metadata = None

    model.to(device)
    model.eval()
    print(f"Loaded model checkpoint: {checkpoint_path}")
    return metadata

def get_record_filename(row, sampling_rate):
    """
    ## Description
    Choose the PTB-XL record path according to the desired sampling rate.

    ## Parameters
    - `row`: a row of the PTB-XL metadata dataframe containing the record information.
    - `sampling_rate`: the desired sampling rate (100 or 500 Hz) for loading the record.
    """

    # PTB-XL stores low-rate and high-rate paths in separate metadata columns
    # This function is an helper to be used to extract the correct path for loading a record at the desired sampling rate
    if sampling_rate == 100:
        return row["filename_lr"]
    if sampling_rate == 500:
        return row["filename_hr"]
    raise ValueError("sampling_rate must be 100 or 500")

def load_ecg_record(row, sampling_rate, data_dir):
    """
    ## Description
    Load one ECG record (WFDB) from PTB-XL metadata row.

    ## Parameters
    - `row`: a row of the PTB-XL metadata dataframe containing the record information.
    - `sampling_rate`: the desired sampling rate (100 or 500 Hz) for loading the record.
    - `data_dir`: the root folder containing PTB-XL metadata and waveform files.
    """

    # Extract the path to the record file according to the desired sampling rate
    rel_path = get_record_filename(row, sampling_rate)

    # WFDB expects a path without extension (it resolves .hea/.dat files).
    full_path = f"{data_dir}/{rel_path}"

    return wfdb.rdrecord(full_path)  # Return the loaded record as a WFDB Record object

def load_all_ecg_records(metadata_df, sampling_rate, data_dir):
    """
    ## Description
    Load all ECG records at a given sampling rate and return a dict keyed by ecg_id.

    ## Parameters
    - `metadata_df`: the PTB-XL metadata dataframe containing information about all records.
    - `sampling_rate`: the desired sampling rate (100 or 500 Hz) for loading the records.
    - `data_dir`: the root folder containing PTB-XL metadata and waveform files.
    """
    records_by_ecg_id = {}

    # Iterate metadata rows and cache each loaded record for quick random access.
    for _, row in metadata_df.iterrows():  # Iterates while exposing both index and row content
        # Take the ecg_id as the key for the loaded record
        ecg_id = int(row["ecg_id"])

        # Load the record using the helper function and store it in the dict as a wfdb Record object
        records_by_ecg_id[ecg_id] = load_ecg_record(row, sampling_rate, data_dir)

    return records_by_ecg_id

def plot_12_leads_two_columns(
    record,
    sampling_rate,
    patient_id,
    ecg_id,
    max_seconds=10.0,
    apply_standard_normalization=False,
):
    """
    ## Description
    Plot 12 ECG leads with 6 leads in the left column and 6 in the right column.

    ## Parameters
    - `record`: a WFDB Record object containing the ECG signal and metadata to plot.
    - `sampling_rate`: the sampling rate of the ECG signal (100 or 500 Hz for PTB-XL).
    - `patient_id`: the patient ID to display in the plot title.
    - `ecg_id`: the ECG ID to display in the plot title.
    - `max_seconds`: the maximum number of seconds to plot (must be <= 10.0).
    - `apply_standard_normalization`: whether to apply standard normalization (z-score) to each lead independently over the plotted window (default: False).
    """

    if max_seconds > 10.0:
        raise ValueError("max_seconds cannot be greater than 10.0")
    if max_seconds <= 0:
        raise ValueError("max_seconds must be greater than 0")

    # Paper-ready font sizes
    title_fs = 13
    label_fs = 12
    tick_fs = 11
    suptitle_fs = 17

    # Extract the signal and lead names from the WFDB Record object
    signal = record.p_signal.copy()
    lead_names = record.sig_name

    # Restrict the visible window to improve readability on long recordings.
    n_samples = min(signal.shape[0], int(max_seconds * sampling_rate))
    t = np.arange(n_samples) / sampling_rate

    if apply_standard_normalization:
        # Normalize each lead independently over the plotted window.
        means = np.mean(signal[:n_samples, :], axis=0)
        stds = np.std(signal[:n_samples, :], axis=0)
        stds[stds == 0] = 1.0
        signal[:n_samples, :] = (signal[:n_samples, :] - means) / stds

    # Share y-axis only when normalized; otherwise keep independent scales.
    fig, axes = plt.subplots(
        6,
        2,
        figsize=(16, 16),
        sharex=True,
        sharey=apply_standard_normalization,
    )

    # Place leads 0..5 in the left column and 6..11 in the right column.
    for lead_idx in range(12):
        col = 0 if lead_idx < 6 else 1
        row = lead_idx if lead_idx < 6 else lead_idx - 6

        ax = axes[row, col]
        ax.plot(t, signal[:n_samples, lead_idx], linewidth=1.0)
        ax.set_title(lead_names[lead_idx], fontsize=title_fs)
        ax.set_ylabel("z-score" if apply_standard_normalization else "mV", fontsize=label_fs)
        ax.tick_params(axis="both", which="both", length=0, labelsize=tick_fs)
        ax.grid(True, alpha=0.7)

    axes[5, 0].set_xlabel("Time [s]", fontsize=label_fs)
    axes[5, 1].set_xlabel("Time [s]", fontsize=label_fs)
    axes[0, 0].set_xlim(0, max_seconds)

    norm_label = "normalized" if apply_standard_normalization else "raw"
    fig.suptitle(
        f"PTB-XL ECG | patient_id={patient_id} | ecg_id={ecg_id} | fs={sampling_rate} Hz | {norm_label}",
        fontsize=suptitle_fs,
    )
    plt.tight_layout()
    save_figure(fig, "12_leads")
    plt.show()

In [ ]:
# Load the entire metadata table (one row per ECG exam).
metadata_path = f"{DATA_DIR}/ptbxl_database.csv"
ptbxl_df = pd.read_csv(metadata_path)

print(f"Metadata shape: {ptbxl_df.shape}")
print("First columns:", list(ptbxl_df.columns[:10]))
display(ptbxl_df.head(3))

In [ ]:
# Choose here: 100 for low-resolution or 500 for high-resolution.
sampling_rate_to_plot = 500

# Choose here: True to standardize each lead (z-score), False to keep raw amplitudes.
apply_standard_normalization = False

# Choose here: how many seconds of ECG to visualize in the final plot (max 10.0).
max_seconds_to_plot = 10.0

# Load all ECG signals once for the selected sampling rate.
all_records = load_all_ecg_records(
    ptbxl_df,
    sampling_rate=sampling_rate_to_plot,
    data_dir=DATA_DIR,
)

print(f"Loaded ECG records: {len(all_records)}")
print(f"Type of each object in the dictionary: {type(all_records[1])}") # Each value in the dict is a WFDB Record object containing the signal and metadata of one ECG exam

In [ ]:
# Set to None to pick a random record (reproducible via SEED), or set to a specific ecg_id integer to pin a record.
# Example: SELECTED_ECG_ID = 42
SELECTED_ECG_ID = 42

if SELECTED_ECG_ID is None:
    rng = np.random.default_rng(SEED)
    selected_idx = int(rng.integers(0, len(ptbxl_df)))
    selected_row = ptbxl_df.iloc[selected_idx]
else:
    selected_row = ptbxl_df[ptbxl_df["ecg_id"] == SELECTED_ECG_ID].iloc[0]
    selected_idx = ptbxl_df.index[ptbxl_df["ecg_id"] == SELECTED_ECG_ID][0]

selected_patient_id = int(selected_row["patient_id"])
selected_ecg_id = int(selected_row["ecg_id"])
record = all_records[selected_ecg_id]

print(f"Selected row index: {selected_idx}")
print(f"Selected patient_id: {selected_patient_id}")
print(f"Selected ecg_id: {selected_ecg_id}")


In [ ]:
# Plot up to the configured number of seconds of the selected 12-lead ECG.
plot_12_leads_two_columns(
    record,
    sampling_rate=sampling_rate_to_plot,
    patient_id=selected_patient_id,
    ecg_id=selected_ecg_id,
    max_seconds=max_seconds_to_plot,
    apply_standard_normalization=apply_standard_normalization,
)

## Signal Filtering: Notch (50 Hz) + Bandpass (0.5-30 Hz)

Apply notch and bandpass filters as in the previous session.

In [ ]:
def apply_notch_filter(ecg_signal, fs, notch_freq=50.0, quality_factor=30.0):
    """
    ## Description
    Apply a notch filter (default 50 Hz) to attenuate power-line interference.
    
    ## Parameters
    - `ecg_signal`: the input ECG signal to filter (2D array: samples x leads).
    - `fs`: the sampling frequency of the ECG signal (e.g., 100 or 500 Hz for PTB-XL).
    - `notch_freq`: the frequency to notch out (default: 50.0 Hz, common power-line interference frequency).
    - `quality_factor`: the quality factor (Q) of the notch filter, controlling the bandwidth around the notch frequency (default: 30.0, higher means narrower notch).
    """
    # iirnotch returns numerator/denominator coefficients of the notch filter.
    b, a = signal.iirnotch(w0=notch_freq, Q=quality_factor, fs=fs)
    # filtfilt applies forward-backward filtering to avoid phase distortion.
    return signal.filtfilt(b, a, ecg_signal, axis=0)

def apply_bandpass_filter(ecg_signal, fs, lowcut=0.5, highcut=30.0, order=4):
    """
    ## Description
    Apply a Butterworth bandpass filter to retain ECG-relevant frequencies.
    
    ## Parameters
    - `ecg_signal`: the input ECG signal to filter (2D array: samples x leads).
    - `fs`: the sampling frequency of the ECG signal (e.g., 100 or 500 Hz for PTB-XL).
    - `lowcut`: the lower cutoff frequency of the bandpass filter (default: 0.5 Hz).
    - `highcut`: the upper cutoff frequency of the bandpass filter (default: 30.0 Hz).
    - `order`: the order of the Butterworth filter (default: 4).
    """
    nyquist = 0.5 * fs
    # Validate cutoff frequencies before filter design.
    if not (0 < lowcut < highcut < nyquist):
        raise ValueError("Bandpass bounds must satisfy 0 < lowcut < highcut < Nyquist")

    # scipy.signal.butter expects normalized cutoffs in [0, 1] relative to Nyquist.
    normalized_low = lowcut / nyquist
    normalized_high = highcut / nyquist
    b, a = signal.butter(order, [normalized_low, normalized_high], btype="bandpass")
    return signal.filtfilt(b, a, ecg_signal, axis=0)

def plot_filtering_steps(
    raw_signal,
    notch_filtered_signal,
    bandpass_filtered_signal,
    fs,
    lead_index=1,
    max_seconds=10.0,
    lead_name="Lead II",
    apply_standard_normalization=False,
):
    """
    ## Description
    Plot one lead across raw, notch-filtered, and bandpass-filtered stages.
    
    ## Parameters
    - `raw_signal`: the original ECG signal before filtering (2D array: samples x leads).
    - `notch_filtered_signal`: the ECG signal after applying the notch filter (same shape as raw_signal).
    - `bandpass_filtered_signal`: the ECG signal after applying the bandpass filter (same shape as raw_signal).
    - `fs`: the sampling frequency of the signals (e.g., 100 or 500 Hz for PTB-XL).
    - `lead_index`: the index of the lead to plot (default: 1 for Lead II).
    - `max_seconds`: the maximum number of seconds to plot (must be <= 10.0).
    """
    # Keep the same safety bound used in previous plotting sections.
    if max_seconds > 10.0:
        raise ValueError("max_seconds cannot be greater than 10.0")
    if max_seconds <= 0:
        raise ValueError("max_seconds must be greater than 0")

    # Paper-ready font sizes (same style as the 12-lead plotting function).
    title_fs = 13
    label_fs = 12
    tick_fs = 11
    suptitle_fs = 16

    # Restrict all stages to the same visible time window.
    n_samples = min(raw_signal.shape[0], int(max_seconds * fs))
    t = np.arange(n_samples) / fs

    # Copy arrays to avoid in-place edits of original signals during optional normalization.
    stages = [
        ("Raw signal", raw_signal.copy()),
        ("After 50 Hz notch filter", notch_filtered_signal.copy()),
        ("After 0.5-30 Hz bandpass filter", bandpass_filtered_signal.copy()),
    ]

    if apply_standard_normalization:
        # Normalize each stage independently on the displayed window.
        normalized_stages = []
        for stage_name, stage_signal in stages:
            means = np.mean(stage_signal[:n_samples, :], axis=0)
            stds = np.std(stage_signal[:n_samples, :], axis=0)
            stds[stds == 0] = 1.0
            stage_signal[:n_samples, :] = (stage_signal[:n_samples, :] - means) / stds
            normalized_stages.append((stage_name, stage_signal))
        stages = normalized_stages

    # Share y-axis only when standardized, otherwise keep independent scales.
    fig, axes = plt.subplots(
        3,
        1,
        figsize=(16, 12),
        sharex=True,
        sharey=apply_standard_normalization,
    )

    # Plot the chosen lead across the three stages with consistent styling.
    for ax, (stage_name, stage_signal) in zip(axes, stages):
        ax.plot(t, stage_signal[:n_samples, lead_index], linewidth=1.1)
        ax.set_ylabel("z-score" if apply_standard_normalization else "Amplitude [mV]", fontsize=label_fs)
        ax.set_title(f"{stage_name} | {lead_name}", fontsize=title_fs)
        ax.grid(True, alpha=0.7)
        ax.tick_params(axis="both", which="both", length=0, labelsize=tick_fs)

    axes[-1].set_xlabel("Time [s]", fontsize=label_fs)
    axes[-1].set_xlim(0, max_seconds)

    norm_label = "normalized" if apply_standard_normalization else "raw"
    fig.suptitle(f"Filtering Stages Comparison | {norm_label}", fontsize=suptitle_fs)
    plt.tight_layout()
    save_figure(fig, "filtering_steps")
    plt.show()

In [ ]:
# Use the selected record already loaded in previous cells.
raw_signal = record.p_signal.copy()
fs = sampling_rate_to_plot

# Step 1: remove narrow-band power-line interference at 50 Hz.
# The function expects a 2D array (samples x leads), so we pass the full 12-lead matrix.
signal_after_notch = apply_notch_filter(
    raw_signal,
    fs=fs,
    notch_freq=50.0,
    quality_factor=30.0,
)

# Step 2: keep ECG morphology while reducing baseline/breathing-related low-frequency motion.
signal_after_bandpass = apply_bandpass_filter(
    signal_after_notch,
    fs=fs,
    lowcut=0.5,
    highcut=30.0,
    order=4,
)

ecg_leads = {
    'I': 0,
    'II': 1,
    'III': 2,
    'aVR': 3,
    'aVL': 4,
    'aVF': 5,
    'V1': 6,
    'V2': 7,
    'V3': 8,
    'V4': 9,
    'V5': 10,
    'V6': 11,
}

# Choose one derivation to visualize intermediate stages
lead_index_to_plot = ecg_leads['III']  # Lead II is commonly used for rhythm analysis and has good SNR
lead_name_to_plot = record.sig_name[lead_index_to_plot]

# Compare raw vs notch-filtered vs bandpass-filtered signal on the same time window.
plot_filtering_steps(
    raw_signal,
    signal_after_notch,
    signal_after_bandpass,
    fs=fs,
    lead_index=lead_index_to_plot,
    max_seconds=max_seconds_to_plot,
    lead_name=lead_name_to_plot,
    apply_standard_normalization=apply_standard_normalization,
)

In [ ]:
# Preprocess every loaded ECG record with the same notch + bandpass pipeline.
# The dictionary keeps direct access by ecg_id for later record-level operations.
bandpassed_records = {}

# The list keeps the same row order as ptbxl_df so it can be stacked into a dataset tensor.
bandpassed_signal_list = []
bandpassed_ecg_ids = []

for _, row in ptbxl_df.iterrows():
    ecg_id = int(row["ecg_id"])
    signal_matrix = all_records[ecg_id].p_signal

    signal_notch = apply_notch_filter(
        signal_matrix,
        fs=sampling_rate_to_plot,
        notch_freq=50.0,
        quality_factor=30.0,
    )
    signal_bandpass = apply_bandpass_filter(
        signal_notch,
        fs=sampling_rate_to_plot,
        lowcut=0.5,
        highcut=30.0,
        order=4,
    )

    bandpassed_records[ecg_id] = signal_bandpass
    bandpassed_signal_list.append(signal_bandpass)
    bandpassed_ecg_ids.append(ecg_id)

# Final tensor shape: (n_records, n_samples, n_leads).
bandpassed_signals = np.stack(bandpassed_signal_list, axis=0)
bandpassed_ecg_ids = np.asarray(bandpassed_ecg_ids, dtype=int)

print(f"Bandpassed dataset shape: {bandpassed_signals.shape}")
print(f"Stored records: {len(bandpassed_records)}")


## Frequency-Domain Check

Verify filtering efficacy via FFT.

In [ ]:
def compute_single_sided_fft(signal_1d, fs):
    """
    ## Description
    Return one-sided FFT frequencies and amplitude spectrum for a 1D signal.

    ## Parameters
    - `signal_1d`: the input signal to analyze (1D array).
    - `fs`: the sampling frequency of the signal (e.g., 100 or 500 Hz for PTB-XL).
    """
    n = len(signal_1d)

    window = np.hanning(n)
    windowed = signal_1d * window

    fft_vals = np.fft.rfft(windowed)
    freqs = np.fft.rfftfreq(n, d=1.0 / fs)

    # Amplitude scaling for one-sided spectrum (window-corrected).
    scale = 2.0 / np.sum(window)
    amplitude = np.abs(fft_vals) * scale
    return freqs, amplitude

def plot_fft_filtering_stages(
    raw_signal,
    notch_filtered_signal,
    bandpass_filtered_signal,
    fs,
    lead_index=1,
    max_freq_to_show=80.0,
    lead_name="Lead II",
):
    """
    ## Description
    Plot FFT magnitude of one lead across raw, notch-filtered, and bandpass-filtered stages.

    ## Parameters
    - `raw_signal`: the original ECG signal before filtering (2D array: samples x leads).
    - `notch_filtered_signal`: the ECG signal after applying the notch filter (same shape as raw_signal).
    - `bandpass_filtered_signal`: the ECG signal after applying the bandpass filter (same shape as raw_signal).
    - `fs`: the sampling frequency of the signals (e.g., 100 or 500 Hz for PTB-XL).
    - `lead_index`: the index of the lead to analyze (default: 1 for Lead II).
    - `max_freq_to_show`: the maximum frequency to display on the x-axis (default: 80.0 Hz).
    - `lead_name`: the name of the lead to display in the plot titles (default: "Lead II").
    """
    stages = [
        ("Raw signal", raw_signal),
        ("After 50 Hz notch filter", notch_filtered_signal),
        ("After 0.5-30 Hz bandpass filter", bandpass_filtered_signal),
    ]

    title_fs = 13
    label_fs = 12
    tick_fs = 11
    suptitle_fs = 16

    fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True, sharey=False)

    for ax, (stage_name, stage_signal) in zip(axes, stages):
        signal_1d = stage_signal[:, lead_index]
        freqs, amp = compute_single_sided_fft(signal_1d, fs)
        ax.plot(freqs, amp, linewidth=1.1)

        ax.set_title(f"{stage_name} | {lead_name}", fontsize=title_fs)
        ax.set_ylabel("Amplitude", fontsize=label_fs)
        ax.tick_params(axis="both", which="both", length=0, labelsize=tick_fs)
        ax.grid(True, alpha=0.7)
        ax.set_xlim(0.0, min(max_freq_to_show, fs / 2.0))

        # Visual guides for the bandpass upper cutoff and powerline frequency.
        ax.axvline(30.0, color="#34495e", linestyle="--", linewidth=1.2, alpha=0.9)
        ax.axvline(50.0, color="#8e44ad", linestyle="--", linewidth=1.2, alpha=0.9)

    axes[-1].set_xlabel("Frequency [Hz]", fontsize=label_fs)
    axes[-1].set_xlim(-0.5, max_freq_to_show + 0.5)
    fig.suptitle("FFT Comparison Across Filtering Stages", fontsize=suptitle_fs)
    plt.tight_layout()
    save_figure(fig, "fft_filtering_stages")
    plt.show()


In [ ]:
# Reuse the already selected lead from the filtering stage visualization.
# If not available, default to Lead II (index 1).
try:
    lead_idx_fft = lead_index_to_plot
except NameError:
    lead_idx_fft = 1

lead_name_fft = record.sig_name[lead_idx_fft]

plot_fft_filtering_stages(
    raw_signal,
    signal_after_notch,
    signal_after_bandpass,
    fs=fs,
    lead_index=lead_idx_fft,
    max_freq_to_show=55.0,
    lead_name=lead_name_fft,
)

## Standard Normalization (Z-score)

Standardize each lead per record to normalize across subjects.

$$
z = \frac{x - \mu}{\sigma}
$$

In [ ]:
# Standardize each record and lead over time: axis=1 is the time axis in (records, samples, leads).
normalization_means = np.mean(bandpassed_signals, axis=1, keepdims=True)
normalization_stds = np.std(bandpassed_signals, axis=1, keepdims=True)

# Numerical safety: if one lead has zero variance, keep denominator at 1 to avoid division by zero.
normalization_stds[normalization_stds == 0] = 1.0

# Final normalized tensor to be used in the following processing/model-preparation steps.
normalized_bandpassed_signals = (bandpassed_signals - normalization_means) / normalization_stds

# Mirror the same normalized content in a dict keyed by ecg_id for quick record-level access.
normalized_bandpassed_records = {
    int(ecg_id): normalized_bandpassed_signals[i]
    for i, ecg_id in enumerate(bandpassed_ecg_ids)
}

# Quick sanity check: mean close to 0 and std close to 1 for a sample record.
sample_idx = 0
sample_mean = np.mean(normalized_bandpassed_signals[sample_idx], axis=0)
sample_std = np.std(normalized_bandpassed_signals[sample_idx], axis=0)

print(f"Normalized dataset shape: {normalized_bandpassed_signals.shape}")
print(f"Stored normalized records: {len(normalized_bandpassed_records)}")
print(f"Sample record per-lead mean (first 3 leads): {np.round(sample_mean[:3], 4)}")
print(f"Sample record per-lead std  (first 3 leads): {np.round(sample_std[:3], 4)}")


In [ ]:
# Plot one normalized 12-lead record to visually confirm amplitude standardization.
# We reuse the same selected ECG id used in previous sections for easy comparison.
normalized_record = normalized_bandpassed_records[selected_ecg_id]

# Restrict visualization window.
n_samples_norm = min(normalized_record.shape[0], int(max_seconds_to_plot * sampling_rate_to_plot))
t_norm = np.arange(n_samples_norm) / sampling_rate_to_plot
lead_names_norm = record.sig_name

title_fs = 13
label_fs = 12
tick_fs = 11
suptitle_fs = 17

fig, axes = plt.subplots(6, 2, figsize=(16, 16), sharex=True, sharey=True)

for lead_idx in range(12):
    col = 0 if lead_idx < 6 else 1
    row = lead_idx if lead_idx < 6 else lead_idx - 6

    ax = axes[row, col]
    ax.plot(t_norm, normalized_record[:n_samples_norm, lead_idx], linewidth=1.0)
    ax.set_title(lead_names_norm[lead_idx], fontsize=title_fs)
    ax.set_ylabel("z-score", fontsize=label_fs)
    ax.tick_params(axis="both", which="both", length=0, labelsize=tick_fs)
    ax.grid(True, alpha=0.7)

axes[5, 0].set_xlabel("Time [s]", fontsize=label_fs)
axes[5, 1].set_xlabel("Time [s]", fontsize=label_fs)
axes[0, 0].set_xlim(0, max_seconds_to_plot)

fig.suptitle(
    f"Normalized ECG (Z-score) | patient_id={selected_patient_id} | ecg_id={selected_ecg_id}",
    fontsize=suptitle_fs,
)
plt.tight_layout()
save_figure(fig, "normalized_12_leads")
plt.show()


## Healthy vs Non-healthy Labeling

Assign binary labels using PTB-XL diagnostic codes: healthy if all records have NORM code only.

In [ ]:
# Build healthy/non-healthy labels from PTB-XL metadata using diagnostic SCP codes.
scp_path = f"{DATA_DIR}/scp_statements.csv"
scp_df = pd.read_csv(scp_path, index_col=0)
diagnostic_codes = set(scp_df.index[scp_df["diagnostic"] == 1])

def parse_scp_codes_field(raw_scp_codes):
    if isinstance(raw_scp_codes, dict):
        return raw_scp_codes
    if pd.isna(raw_scp_codes):
        return {}
    return ast.literal_eval(raw_scp_codes)

def is_healthy_record_strict(raw_scp_codes):
    parsed = parse_scp_codes_field(raw_scp_codes)
    record_codes = set(parsed.keys())
    diagnostic_in_record = record_codes.intersection(diagnostic_codes)
    return diagnostic_in_record == {"NORM"}

# Record-level label.
ptbxl_labeled_df = ptbxl_df.copy()
ptbxl_labeled_df["is_healthy_record"] = ptbxl_labeled_df["scp_codes"].apply(is_healthy_record_strict)

# Patient-level label: healthy only if all available records are healthy.
patient_labels_df = (
    ptbxl_labeled_df.groupby("patient_id", as_index=False)
    .agg(
        is_healthy_patient=("is_healthy_record", "all"),
        n_records=("ecg_id", "count"),
    )
)

# Report the index of healthy and non-healthy patients in the current subset
healthy_patients = patient_labels_df.loc[patient_labels_df["is_healthy_patient"], "patient_id"].tolist()
nonhealthy_patients = patient_labels_df.loc[~patient_labels_df["is_healthy_patient"], "patient_id"].tolist()

if len(healthy_patients) == 0 or len(nonhealthy_patients) == 0:
    raise RuntimeError("Could not find both healthy and non-healthy patients in the current subset.")

# Pick one representative ECG per class, ensuring it is available in the normalized dictionary.
healthy_row = ptbxl_labeled_df[
    (ptbxl_labeled_df["patient_id"] == healthy_patients[0])
    & (ptbxl_labeled_df["ecg_id"].isin(normalized_bandpassed_records.keys()))
].iloc[0]

nonhealthy_row = ptbxl_labeled_df[
    (ptbxl_labeled_df["patient_id"] == nonhealthy_patients[0])
    & (ptbxl_labeled_df["ecg_id"].isin(normalized_bandpassed_records.keys()))
].iloc[0]

healthy_patient_id = int(healthy_row["patient_id"])
healthy_ecg_id = int(healthy_row["ecg_id"])
nonhealthy_patient_id = int(nonhealthy_row["patient_id"])
nonhealthy_ecg_id = int(nonhealthy_row["ecg_id"])

print(f"Healthy patient selected: patient_id={healthy_patient_id}, ecg_id={healthy_ecg_id}")
print(f"Non-healthy patient selected: patient_id={nonhealthy_patient_id}, ecg_id={nonhealthy_ecg_id}")
print(patient_labels_df["is_healthy_patient"].value_counts().rename(index={True: 'Healthy', False: 'Non-healthy'}))


In [ ]:
# Full 12-lead comparison as two separate figures (healthy and non-healthy).
healthy_signal_12 = normalized_bandpassed_records[healthy_ecg_id]
nonhealthy_signal_12 = normalized_bandpassed_records[nonhealthy_ecg_id]

n_samples_compare_12 = min(
    healthy_signal_12.shape[0],
    nonhealthy_signal_12.shape[0],
    int(max_seconds_to_plot * sampling_rate_to_plot),
)
t_compare_12 = np.arange(n_samples_compare_12) / sampling_rate_to_plot
lead_names_12 = all_records[healthy_ecg_id].sig_name

# Colorblind-safe class colors (Okabe-Ito inspired).
HEALTHY_COLOR = "#0072B2"
NONHEALTHY_COLOR = "#D55E00"

# Extract pathology text for the selected non-healthy ECG from SCP diagnostic codes.
nonhealthy_codes_dict = parse_scp_codes_field(nonhealthy_row["scp_codes"])
nonhealthy_diagnostic_codes = sorted(
    set(nonhealthy_codes_dict.keys()).intersection(diagnostic_codes).difference({"NORM"})
)

if len(nonhealthy_diagnostic_codes) == 0:
    nonhealthy_pathology_text = "No diagnostic pathology code available"
else:
    pathology_entries = []
    for code in nonhealthy_diagnostic_codes:
        if code in scp_df.index:
            pathology_entries.append(f"{code}: {scp_df.loc[code, 'description']}")
        else:
            pathology_entries.append(code)
    nonhealthy_pathology_text = "; ".join(pathology_entries)

def plot_single_subject_12_leads(signal_matrix, line_color, suptitle, output_suffix):
    fig, axes = plt.subplots(6, 2, figsize=(16, 16), sharex=True, sharey=True)

    for lead_idx in range(12):
        col = 0 if lead_idx < 6 else 1
        row = lead_idx if lead_idx < 6 else lead_idx - 6

        ax = axes[row, col]
        ax.plot(
            t_compare_12,
            signal_matrix[:n_samples_compare_12, lead_idx],
            color=line_color,
            linewidth=1.0,
        )
        ax.set_title(lead_names_12[lead_idx], fontsize=12)
        ax.set_ylabel("z-score", fontsize=11)
        ax.grid(True, alpha=0.7)
        ax.tick_params(axis="both", which="both", length=0, labelsize=10)

    axes[5, 0].set_xlabel("Time [s]", fontsize=11)
    axes[5, 1].set_xlabel("Time [s]", fontsize=11)
    axes[0, 0].set_xlim(0, max_seconds_to_plot)

    fig.suptitle(suptitle, fontsize=14)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    save_figure(fig, output_suffix)
    plt.show()

plot_single_subject_12_leads(
    healthy_signal_12,
    line_color=HEALTHY_COLOR,
    suptitle=f"Processed + Normalized ECG (Healthy) | patient_id={healthy_patient_id} | ecg_id={healthy_ecg_id}",
    output_suffix="normalized_healthy_12leads",
)

plot_single_subject_12_leads(
    nonhealthy_signal_12,
    line_color=NONHEALTHY_COLOR,
    suptitle=(
        f"Processed + Normalized ECG (Non-healthy) | patient_id={nonhealthy_patient_id} | ecg_id={nonhealthy_ecg_id}\n"
        f"Pathology: {nonhealthy_pathology_text}"
    ),
    output_suffix="normalized_nonhealthy_12leads",
)

print(f"Non-healthy pathology: {nonhealthy_pathology_text}")


In [ ]:
# Plot processed+normalized signals for one healthy and one non-healthy patient.
# Lead II is a common choice for rhythm/morphology inspection.
lead_name_to_compare = "II"
lead_idx_compare = ecg_leads[lead_name_to_compare]

healthy_signal = normalized_bandpassed_records[healthy_ecg_id]
nonhealthy_signal = normalized_bandpassed_records[nonhealthy_ecg_id]

# Colorblind-safe class colors (Okabe-Ito inspired).
HEALTHY_COLOR = "#0072B2"
NONHEALTHY_COLOR = "#D55E00"

n_samples_compare = min(
    healthy_signal.shape[0],
    nonhealthy_signal.shape[0],
    int(max_seconds_to_plot * sampling_rate_to_plot),
)
t_compare = np.arange(n_samples_compare) / sampling_rate_to_plot

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True, sharey=True)

axes[0].plot(t_compare, healthy_signal[:n_samples_compare, lead_idx_compare], color=HEALTHY_COLOR, linewidth=1.1)
axes[0].set_title(
    f"Healthy | patient_id={healthy_patient_id} | ecg_id={healthy_ecg_id} | lead {lead_name_to_compare}",
    fontsize=12,
)
axes[0].set_ylabel("z-score", fontsize=11)
axes[0].grid(True, alpha=0.7)
axes[0].tick_params(axis="both", which="both", length=0, labelsize=10)

axes[1].plot(t_compare, nonhealthy_signal[:n_samples_compare, lead_idx_compare], color=NONHEALTHY_COLOR, linewidth=1.1)
axes[1].set_title(
    f"Non-healthy | patient_id={nonhealthy_patient_id} | ecg_id={nonhealthy_ecg_id} | lead {lead_name_to_compare}",
    fontsize=12,
)
axes[1].set_ylabel("z-score", fontsize=11)
axes[1].set_xlabel("Time [s]", fontsize=11)
axes[1].grid(True, alpha=0.7)
axes[1].tick_params(axis="both", which="both", length=0, labelsize=10)

axes[0].set_xlim(0, max_seconds_to_plot)
fig.suptitle("Processed + Normalized ECG Comparison", fontsize=14)
plt.tight_layout()
save_figure(fig, "normalized_healthy_vs_nonhealthy_leadII")
plt.show()


## Time- and Frequency-Domain HRV Features

Extract HRV features (mean HR, SDNN, RMSSD, pNN50, LF, HF, LF/HF) from Lead II R-R intervals.

In [ ]:
# Lead II is the standard clinical choice for R-peak detection (good P-QRS-T morphology and SNR).
LEAD_HRV = ecg_leads["II"]

def compute_rr_intervals(signal_1d, fs):
    """
    Detect R peaks with NeuroKit2 and return:
    - rr_sec: R-R intervals in seconds
    - r_peaks: R peak sample indices
    Returns (None, None) if too few peaks are detected.
    """
    cleaned = nk.ecg_clean(signal_1d, sampling_rate=fs)
    _, info = nk.ecg_peaks(cleaned, sampling_rate=fs)
    r_peaks = info["ECG_R_Peaks"]
    if len(r_peaks) < 3:
        return None, None
    rr_sec = np.diff(r_peaks) / fs
    return rr_sec, r_peaks

def compute_hrv_time_features(rr_sec):
    """
    Compute time-domain HRV features from R-R intervals (in seconds).

    Returns
    -------
    mean_hr_bpm : float
    sdnn_ms     : float
    rmssd_ms    : float
    pnn50_pct   : float
    """
    mean_rr = np.mean(rr_sec)
    mean_hr_bpm = 60.0 / mean_rr

    rr_ms = rr_sec * 1000.0
    sdnn_ms = np.std(rr_ms, ddof=1)

    successive_diffs_ms = np.diff(rr_ms)
    rmssd_ms = np.sqrt(np.mean(successive_diffs_ms ** 2))

    if len(successive_diffs_ms) == 0:
        pnn50_pct = np.nan
    else:
        pnn50_pct = 100.0 * np.mean(np.abs(successive_diffs_ms) > 50.0)

    return mean_hr_bpm, sdnn_ms, rmssd_ms, pnn50_pct

def compute_hrv_freq_features(rr_sec, r_peaks, fs, interp_fs=4.0):
    """
    Compute LF, HF, and LF/HF from RR intervals.

    Steps:
    1) Build beat times from detected R peaks.
    2) Interpolate RR series on an evenly sampled grid.
    3) Estimate PSD via Welch and integrate LF/HF bands.
    """
    if len(rr_sec) < 4 or r_peaks is None:
        return np.nan, np.nan, np.nan

    rr_ms = rr_sec * 1000.0

    # RR interval i is associated with the second beat time of the pair.
    rr_times_sec = r_peaks[1:] / fs

    if len(rr_times_sec) < 4:
        return np.nan, np.nan, np.nan

    t_min, t_max = rr_times_sec[0], rr_times_sec[-1]
    if t_max <= t_min:
        return np.nan, np.nan, np.nan

    t_uniform = np.arange(t_min, t_max, 1.0 / interp_fs)
    if len(t_uniform) < 8:
        return np.nan, np.nan, np.nan

    rr_uniform_ms = np.interp(t_uniform, rr_times_sec, rr_ms)
    rr_uniform_ms = signal.detrend(rr_uniform_ms)

    nperseg = min(256, len(rr_uniform_ms))
    if nperseg < 8:
        return np.nan, np.nan, np.nan

    freqs, psd = signal.welch(rr_uniform_ms, fs=interp_fs, nperseg=nperseg)

    lf_mask = (freqs >= 0.04) & (freqs < 0.15)
    hf_mask = (freqs >= 0.15) & (freqs <= 0.40)

    lf_power = np.trapezoid(psd[lf_mask], freqs[lf_mask]) if np.any(lf_mask) else np.nan
    hf_power = np.trapezoid(psd[hf_mask], freqs[hf_mask]) if np.any(hf_mask) else np.nan

    if np.isnan(lf_power) or np.isnan(hf_power) or hf_power <= 0:
        lf_hf_ratio = np.nan
    else:
        lf_hf_ratio = lf_power / hf_power

    return lf_power, hf_power, lf_hf_ratio

# ------------------------------------------------------------------
# Compute HRV features for every record in the normalized dataset.
# ------------------------------------------------------------------
records_hrv = []

for ecg_id, signal_matrix in normalized_bandpassed_records.items():
    signal_1d = signal_matrix[:, LEAD_HRV]
    rr_intervals, r_peaks = compute_rr_intervals(signal_1d, fs=sampling_rate_to_plot)

    if rr_intervals is None:
        print(f"  [WARN] ecg_id={ecg_id}: insufficient R peaks detected, skipping.")
        continue

    mean_hr_bpm, sdnn_ms, rmssd_ms, pnn50_pct = compute_hrv_time_features(rr_intervals)
    lf_power, hf_power, lf_hf_ratio = compute_hrv_freq_features(
        rr_intervals,
        r_peaks,
        fs=sampling_rate_to_plot,
        interp_fs=4.0,
    )

    patient_id = int(
        ptbxl_labeled_df.loc[ptbxl_labeled_df["ecg_id"] == ecg_id, "patient_id"].iloc[0]
    )

    records_hrv.append(
        {
            "ecg_id": ecg_id,
            "patient_id": patient_id,
            "mean_hr_bpm": mean_hr_bpm,
            "sdnn_ms": sdnn_ms,
            "rmssd_ms": rmssd_ms,
            "pnn50_pct": pnn50_pct,
            "lf_power": lf_power,
            "hf_power": hf_power,
            "lf_hf_ratio": lf_hf_ratio,
        }
    )

hrv_records_df = pd.DataFrame(records_hrv)

# ------------------------------------------------------------------
# Aggregate at patient level: average feature values across records.
# ------------------------------------------------------------------
patient_hrv_df = (
    hrv_records_df.groupby("patient_id", as_index=False)
    .agg(
        mean_hr_bpm=("mean_hr_bpm", "mean"),
        sdnn_ms=("sdnn_ms", "mean"),
        rmssd_ms=("rmssd_ms", "mean"),
        pnn50_pct=("pnn50_pct", "mean"),
        lf_power=("lf_power", "mean"),
        hf_power=("hf_power", "mean"),
        lf_hf_ratio=("lf_hf_ratio", "mean"),
    )
)

# Attach the health label so features and class are always aligned.
patient_hrv_df = patient_hrv_df.merge(
    patient_labels_df[["patient_id", "is_healthy_patient"]],
    on="patient_id",
    how="left",
)

feature_cols = [
    "mean_hr_bpm",
    "sdnn_ms",
    "rmssd_ms",
    "pnn50_pct",
    "lf_power",
    "hf_power",
    "lf_hf_ratio",
]

print(f"HRV features computed for {len(hrv_records_df)} records / {len(patient_hrv_df)} patients")
print(f"\nPer-class summary (mean +/- std):")
display(
    patient_hrv_df.groupby("is_healthy_patient")[feature_cols]
    .agg(["mean", "std"])
    .round(2)
    .rename(index={True: "Healthy", False: "Non-healthy"})
)

## Patient Demographics and Missing Value Analysis

Extract demographic features (age, sex, height, weight) at patient level. Missing values are clinically and statistically significant: absence of data can indicate measurement failures, patient refusal, or specific inclusion/exclusion criteria.

In [ ]:
# Extract demographic information at patient level.
demographic_cols = ["age", "sex", "height", "weight"]

# Build patient-level demographics dataframe: row per patient, aggregating multiple records.
patient_demo_records = []

for patient_id in ptbxl_labeled_df["patient_id"].unique():
    patient_data = ptbxl_labeled_df[ptbxl_labeled_df["patient_id"] == patient_id]
    
    # Since demographics should be same per patient, take first non-NaN value or NaN if all are NaN.
    demo_row = {"patient_id": patient_id}
    for col in demographic_cols:
        values = patient_data[col].dropna()
        demo_row[col] = values.iloc[0] if len(values) > 0 else np.nan
    
    demo_row["n_records"] = len(patient_data)
    patient_demo_records.append(demo_row)

patient_demo_df = pd.DataFrame(patient_demo_records)

# Attach health labels.
patient_demo_df = patient_demo_df.merge(
    patient_labels_df[["patient_id", "is_healthy_patient"]], 
    on="patient_id", 
    how="left"
)

print(f"Patient demographics dataset: {patient_demo_df.shape[0]} patients"), print(f"Available columns: {list(patient_demo_df.columns)}"), print()

In [ ]:
# NaN analysis: quantify missing data.
print("=" * 80)
print("MISSING VALUE ANALYSIS - Patient Demographics")
print("=" * 80)

nan_summary = []
for col in demographic_cols:
    n_missing = patient_demo_df[col].isna().sum()
    pct_missing = 100.0 * n_missing / len(patient_demo_df)
    n_present = len(patient_demo_df) - n_missing
    nan_summary.append({
        "Feature": col,
        "Present": n_present,
        "Missing": n_missing,
        "Missing %": f"{pct_missing:.1f}%"
    })

nan_df = pd.DataFrame(nan_summary)
print("\nMissingness per demographic feature:")
print(nan_df.to_string(index=False))
print()

# Correlation of missingness: are some features missing together?
missing_pattern = patient_demo_df[demographic_cols].isna().astype(int)
print("\nMissing data patterns (count of patients):")
missing_pattern_counts = missing_pattern.value_counts().sort_values(ascending=False)
display(missing_pattern_counts.head(10))

# Summary statistics for present values (only non-NaN).
print("\nDescriptive statistics (only non-missing values):")
display(patient_demo_df[demographic_cols].describe().round(2))

In [ ]:
# Visualize missingness by health class.

# Colorblind-safe class colors (Okabe-Ito inspired).
HEALTHY_COLOR = "#0072B2"
NONHEALTHY_COLOR = "#D55E00"

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left plot: missingness by feature and health class
for ax, (health_label, health_bool, bar_color, bar_hatch) in zip(
    axes,
    [
        ("Healthy", True, HEALTHY_COLOR, "//"),
        ("Non-healthy", False, NONHEALTHY_COLOR, ".."),
    ],
):
    subset = patient_demo_df[patient_demo_df["is_healthy_patient"] == health_bool]
    missing_rates = []
    for col in demographic_cols:
        missing_pct = 100.0 * subset[col].isna().sum() / len(subset)
        missing_rates.append(missing_pct)

    bars = ax.bar(demographic_cols, missing_rates, color=bar_color, edgecolor="#1f2937", linewidth=0.8)
    for bar in bars:
        bar.set_hatch(bar_hatch)

    ax.set_ylabel("Missing [%]", fontsize=11)
    ax.set_title(f"Missingness: {health_label} Patients", fontsize=12)
    ax.set_ylim(0, 100)
    ax.grid(True, alpha=0.3, axis="y")
    for i, rate in enumerate(missing_rates):
        ax.text(i, rate + 2, f"{rate:.1f}%", ha="center", fontsize=10)

plt.tight_layout()
save_figure(fig, "demographic_missingness_by_class")
plt.show()

print(f"\nVisualization saved showing missingness patterns across health classes.")
print(f"Total patients with complete demographic data: {(~patient_demo_df[demographic_cols].isna().any(axis=1)).sum()} / {len(patient_demo_df)}")


## Record Windowing and Overlap-Based Augmentation

In this final preprocessing step, each full ECG record is converted into many shorter training samples (windows).
This increases the number of model inputs while preserving the original label semantics (healthy vs non-healthy).

### Why windowing helps
- Neural networks often train better with fixed-length inputs.
- Overlapping windows increase dataset size without collecting new patients.
- Local windows make it easier for the model to focus on morphology in short temporal segments.
- The 0.5 s step introduces overlap, which acts as temporal data augmentation.

### Window definition used in this notebook
- Window duration: **1.0 s**
- Step (hop): **0.5 s**

Since the step is smaller than the window length, consecutive windows overlap by:
$$
T_{overlap} = T_{window} - T_{step} = 1.0 - 0.5 = 0.5\ \text{s}
$$

For sampling rate $f_s$, each sample contains:
$$
N_{window} = T_{window}\cdot f_s
$$

### Label propagation rule
Each generated window inherits the patient-level binary label already computed before:
- `1` = healthy
- `0` = non-healthy

This section builds both:
- a tensor for model training
- a metadata table to track where each window came from

In [ ]:
# ------------------------------------------------------------
# Record windowing for overlap-based data augmentation
# ------------------------------------------------------------
WINDOW_SEC = 1.0
STEP_SEC = 0.5

def extract_windows(signal_matrix, fs, window_sec=1.0, step_sec=0.5):
    """
    Convert one ECG record (samples x leads) into overlapping fixed-length windows.

    Returns
    -------
    windows : np.ndarray of shape (n_windows, window_samples, n_leads)
    starts_sec : np.ndarray of shape (n_windows,)
        Start time (in seconds) of each window in the original signal.
    """
    if signal_matrix.ndim != 2:
        raise ValueError("signal_matrix must have shape (n_samples, n_leads)")

    n_samples, n_leads = signal_matrix.shape
    window_samples = int(round(window_sec * fs))
    step_samples = int(round(step_sec * fs))

    if window_samples <= 0:
        raise ValueError("window_sec must produce at least 1 sample")
    if step_samples <= 0:
        raise ValueError("step_sec must produce at least 1 sample")

    # If record is shorter than one window, keep one zero-padded/truncated-free sample from start.
    if n_samples <= window_samples:
        start_positions = np.array([0], dtype=int)
    else:
        # Build the array of window start positions with the given step, ensuring we cover the entire signal.
        start_positions = np.arange(0, n_samples - window_samples + 1, step_samples, dtype=int)
        # Include the last admissible start to cover the tail segment.
        last_valid_start = n_samples - window_samples
        if start_positions[-1] != last_valid_start:
            start_positions = np.append(start_positions, last_valid_start)

    windows = np.empty((len(start_positions), window_samples, n_leads), dtype=signal_matrix.dtype)
    starts_sec = np.empty(len(start_positions), dtype=float)

    for i, start in enumerate(start_positions):
        # Find the end index for the current window, ensuring it does not exceed signal length.
        end = start + window_samples
        # Individuate the window slice, which will automatically handle cases where end > n_samples by slicing up to n_samples.
        windows[i] = signal_matrix[start:end, :]
        # Retrieve the actual start time in seconds for this window.
        starts_sec[i] = start / fs

    return windows, starts_sec

print("=" * 90)
print("WINDOW CONFIGURATION")
print("=" * 90)
print(f"Sampling rate: {sampling_rate_to_plot} Hz")
print(f"Window length: {WINDOW_SEC:.2f} s ({int(round(WINDOW_SEC * sampling_rate_to_plot))} samples)")
print(f"Step size    : {STEP_SEC:.2f} s ({int(round(STEP_SEC * sampling_rate_to_plot))} samples)")
print(f"Overlap      : {WINDOW_SEC - STEP_SEC:.2f} s")
print()

In [ ]:
# Lookups to map ecg_id -> patient_id and patient_id -> label.

# Create a dictionary where each ecg_id is the key and the corresponding patient_id is the value.
ecg_to_patient = ptbxl_labeled_df.set_index("ecg_id")["patient_id"].astype(int).to_dict()

# Create a dictionary where each patient_id is the key and the corresponding is_healthy_patient boolean label is the value.
patient_to_health = patient_labels_df.set_index("patient_id")["is_healthy_patient"].astype(bool).to_dict()

windows_list = []
labels_list = []
window_ecg_ids = []
window_patient_ids = []
window_start_secs = []
per_record_window_count = []

# Loop on each normalized record (dictionary with ecg_id keys), extract windows, and assign the same label to all windows from the same record.
for ecg_id, signal_matrix in normalized_bandpassed_records.items():
    patient_id = int(ecg_to_patient[ecg_id])
    is_healthy_patient = bool(patient_to_health[patient_id])
    binary_label = 1 if is_healthy_patient else 0  # 1=healthy, 0=non-healthy

    # Extract overlapping windows from the current record's signal matrix.
    record_windows, record_starts_sec = extract_windows(
        signal_matrix,
        fs=sampling_rate_to_plot,
        window_sec=WINDOW_SEC,
        step_sec=STEP_SEC,
    )

    # Store the number of windows generated from this record for reporting.
    n_record_windows = record_windows.shape[0]
    per_record_window_count.append((int(ecg_id), n_record_windows))

    # Generate metadata for each window and append to the global lists.
    windows_list.append(record_windows)
    # Append as much labels as windows generated from this record, ensuring alignment between features and labels.
    labels_list.extend([binary_label] * n_record_windows)
    window_ecg_ids.extend([int(ecg_id)] * n_record_windows)
    window_patient_ids.extend([patient_id] * n_record_windows)
    window_start_secs.extend(record_starts_sec.tolist())

# Final tensors ready for model input and supervised learning.
X_windows = np.concatenate(windows_list, axis=0)
y_windows = np.asarray(labels_list, dtype=np.int64)

# Store window-level metadata in a DataFrame for easy analysis and potential future use in modeling or interpretation.
window_metadata_df = pd.DataFrame(
    {
        "ecg_id": np.asarray(window_ecg_ids, dtype=int),
        "patient_id": np.asarray(window_patient_ids, dtype=int),
        "window_start_sec": np.asarray(window_start_secs, dtype=float),
        "label_is_healthy": y_windows.astype(bool),
    }
)

# Report the shapes and class distribution at window level, as well as the number of windows generated per record.
window_duration_sec = WINDOW_SEC
record_window_count_df = pd.DataFrame(per_record_window_count, columns=["ecg_id", "n_windows"])

print("=" * 90)
print("WINDOWING OUTPUT SUMMARY")
print("=" * 90)
print(f"X_windows shape: {X_windows.shape} (n_windows, n_samples, n_leads)")
print(f"y_windows shape: {y_windows.shape}")
print(f"Window duration: {window_duration_sec:.2f} s")
print()

print("Class distribution at window level:")
display(
    pd.Series(y_windows).value_counts().sort_index()
    .rename(index={0: "Non-healthy", 1: "Healthy"})
    .rename("n_windows")
)

print("Per-record number of generated windows (first 10 records):")
display(record_window_count_df.head(10))

print("Window metadata preview (first 5 windows):")
display(window_metadata_df.head(5))

In [ ]:
# Optional quick check for one ECG record: expected vs observed approximate window count.
# For a long record, approximate count ~ floor((N - W)/S) + 1 (+ tail inclusion if needed).
example_ecg_id = int(window_metadata_df.iloc[0]["ecg_id"])

example_n_samples = normalized_bandpassed_records[example_ecg_id].shape[0]
W = int(round(WINDOW_SEC * sampling_rate_to_plot))
S = int(round(STEP_SEC * sampling_rate_to_plot))
if example_n_samples <= W:
    approx_expected = 1
else:
    approx_expected = int(np.floor((example_n_samples - W) / S) + 1)
observed = int(record_window_count_df.loc[record_window_count_df["ecg_id"] == example_ecg_id, "n_windows"].iloc[0])
print(
    f"Example ecg_id={example_ecg_id}: approx expected windows={approx_expected}, observed={observed}"
)

## Visual Check of Window Overlap

To make the augmentation mechanism intuitive, we visualize multiple consecutive windows from the same ECG record.

Because each window is 1.0 s and the step is 0.5 s, adjacent windows overlap by 0.5 s.
This plot illustrates how temporal overlap creates additional training samples.

In [ ]:
# Visualize overlap across consecutive windows from the same ECG record
# in 4 subfigures so the temporal shift is clearer.

def plot_consecutive_windows_subfigures(
    X_windows,
    window_metadata_df,
    normalized_bandpassed_records,
    fs,
    window_sec=1.0,
    lead_idx=1,
    lead_name="Lead II",
    n_windows_to_plot=4,
    ecg_id=None,
):
    if ecg_id is None:
        ecg_id = int(window_metadata_df.iloc[0]["ecg_id"])

    # Filter the metadata to get windows from the selected ECG record, sorted by start time.
    subset = window_metadata_df[window_metadata_df["ecg_id"] == ecg_id].sort_values("window_start_sec")
    if len(subset) == 0:
        raise ValueError("No windows found for selected ecg_id")

    n_plot = min(n_windows_to_plot, len(subset))
    selected_idx = subset.index[:n_plot].to_list()

    # Keep same absolute-time axis limits in all panels to highlight window motion.
    first_start = float(window_metadata_df.loc[selected_idx[0], "window_start_sec"])
    last_start = float(window_metadata_df.loc[selected_idx[-1], "window_start_sec"])
    x_min = max(0.0, first_start - 0.05)
    x_max = last_start + window_sec + 0.05

    # Reference full ECG lead from the same record (background in every panel).
    reference_signal = normalized_bandpassed_records[int(ecg_id)][:, lead_idx]
    t_ref = np.arange(len(reference_signal)) / fs
    ref_mask = (t_ref >= x_min) & (t_ref <= x_max)

    # Extract the portion of the reference signal that will be visible in the plot to avoid plotting the entire trace.
    t_ref_vis = t_ref[ref_mask]
    ref_vis = reference_signal[ref_mask]

    fig, axes = plt.subplots(n_plot, 1, figsize=(14, 2.8 * n_plot), sharex=True, sharey=True)
    if n_plot == 1:
        axes = [axes]

    colors = plt.get_cmap("viridis")(np.linspace(0.15, 0.95, n_plot))

    for k, (ax, idx) in enumerate(zip(axes, selected_idx), start=1):
        w = X_windows[idx][:, lead_idx]
        start_t = float(window_metadata_df.loc[idx, "window_start_sec"])
        t_abs = start_t + np.arange(len(w)) / fs
        end_t = start_t + window_sec

        # Plot the full reference ECG trace to preserve global context.
        ax.plot(t_ref_vis, ref_vis, linewidth=1.0, color="#7f8c8d", alpha=0.45, label="Reference ECG")

        # Highlight and overlay the active window.
        ax.axvspan(start_t, end_t, color=colors[k - 1], alpha=0.20)
        ax.plot(t_abs, w, linewidth=1.8, color=colors[k - 1], label="Active window")
        ax.axvline(start_t, color="#2c3e50", linestyle="--", linewidth=1.0, alpha=0.6)
        ax.axvline(end_t, color="#2c3e50", linestyle="--", linewidth=1.0, alpha=0.6)

        ax.set_ylabel("z-score")
        ax.grid(True, alpha=0.3)
        ax.set_title(f"Window {k}: start={start_t:.2f}s, end={end_t:.2f}s", fontsize=11)
        ax.set_xlim(x_min, x_max)

        if k == 1:
            ax.legend(loc="upper right")

    axes[-1].set_xlabel("Absolute time [s]")
    fig.suptitle(f"Consecutive Overlapping Windows (4 subfigures) | ecg_id={ecg_id} | {lead_name}", fontsize=13)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    return fig, axes

target_ecg_id = int(window_metadata_df.iloc[0]["ecg_id"])
fig, _ = plot_consecutive_windows_subfigures(
    X_windows=X_windows,
    window_metadata_df=window_metadata_df,
    normalized_bandpassed_records=normalized_bandpassed_records,
    fs=sampling_rate_to_plot,
    window_sec=WINDOW_SEC,
    lead_idx=1,
    lead_name="Lead II",
    n_windows_to_plot=4,
    ecg_id=target_ecg_id,
)
save_figure(fig, "window_overlap_consecutive_subfigures")
plt.show()

print("Plotted 4 subfigures with the reference ECG visible in each panel.")

## Train and Test Set Preparation

With windows extracted and HRV/demographic features computed, we now split the dataset into training and test sets using **patient-level stratification** to prevent data leakage. 

Three neural network input architectures are created as dictionaries:
1. **Signals only**: raw ECG windows
2. **Signals + HRV metrics**: combined with heart-rate variability features
3. **Signals + Demographics**: combined with age, sex, height, weight

All tabular features are imputed (median fill) and standardized using statistics computed only on the training set.

In [ ]:
# ------------------------------------------------------------------
# Build patient-wise train/test sets as dictionaries for 3 architectures
# ------------------------------------------------------------------

# 1) Patient-level multimodal table (label + HRV + demographics)
hrv_cols = feature_cols.copy()
demo_cols = demographic_cols.copy()

patient_multimodal_df = (
    patient_labels_df[["patient_id", "is_healthy_patient"]]
    .drop_duplicates(subset=["patient_id"])
    .merge(patient_hrv_df[["patient_id"] + hrv_cols], on="patient_id", how="left")
    .merge(patient_demo_df[["patient_id"] + demo_cols], on="patient_id", how="left")
)

patient_multimodal_df["label"] = patient_multimodal_df["is_healthy_patient"].astype(int)


In [ ]:
# 2) Patient-wise split (prevents leakage across windows from same patient)
patient_ids_all = patient_multimodal_df["patient_id"].to_numpy()
patient_labels_all = patient_multimodal_df["label"].to_numpy()

train_patient_ids, test_patient_ids = train_test_split(
    patient_ids_all,
    test_size=0.2,
    random_state=SEED,
    stratify=patient_labels_all,
)

# Separate patient ids to be sure that the train/test split is done at patient level and not at window level
# Otherwise, the split could have caused data leakage by having windows from the same patient in both train and test sets
train_patient_ids = np.asarray(train_patient_ids, dtype=int)
test_patient_ids = np.asarray(test_patient_ids, dtype=int)


In [ ]:
# 3) Imputation and scaling for tabular features (fit on train patients only)
all_tabular_cols = hrv_cols + demo_cols

train_patient_df = patient_multimodal_df[patient_multimodal_df["patient_id"].isin(train_patient_ids)].copy()
test_patient_df = patient_multimodal_df[patient_multimodal_df["patient_id"].isin(test_patient_ids)].copy()

# Fill NaN values with the median computed on the training set to avoid data leakage, then standardize using train set statistics.
# This is only one of many possible imputation strategies, chosen for simplicity and robustness in small datasets.
impute_values = train_patient_df[all_tabular_cols].median(numeric_only=True)
train_patient_df[all_tabular_cols] = train_patient_df[all_tabular_cols].fillna(impute_values)
test_patient_df[all_tabular_cols] = test_patient_df[all_tabular_cols].fillna(impute_values)

tab_mean = train_patient_df[all_tabular_cols].mean()
tab_std = train_patient_df[all_tabular_cols].std().replace(0, 1.0)

# Standardize features to have zero mean and unit variance based on training set statistics, ensuring the same transformation is applied to the test set without data leakage
train_patient_df[all_tabular_cols] = (train_patient_df[all_tabular_cols] - tab_mean) / tab_std
test_patient_df[all_tabular_cols] = (test_patient_df[all_tabular_cols] - tab_mean) / tab_std


In [ ]:
# 4) Window-level masks and helper to map patient features to each window
window_patient_ids = window_metadata_df["patient_id"].to_numpy(dtype=int)

# Create boolean masks to identify which windows belong to train patients and which belong to test patients, ensuring that all windows from the same patient are assigned to the same set without leakage
train_mask = np.isin(window_patient_ids, train_patient_ids)
test_mask = np.isin(window_patient_ids, test_patient_ids)

print(train_mask.sum(), "windows in train set")
print(test_mask.sum(), "windows in test set")

# Extract the corresponding metadata for the train and test windows, which can be useful for analysis or as input to models that combine window-level features with patient-level features
train_meta = window_metadata_df.loc[train_mask].reset_index(drop=True)
test_meta = window_metadata_df.loc[test_mask].reset_index(drop=True)

def map_patient_features_to_windows(meta_df, patient_df, cols):
    # Retrieve the patient-level features for the windows in meta_df by matching patient_id, ensuring the order of rows in the output matches the order of windows in meta_df
    lookup = patient_df.set_index("patient_id")[cols]
    # Return a dataframe of shape (n_windows, n_features) where each row corresponds to the patient-level features for the patient associated with that window,
    # ensuring the correct alignment between windows and their corresponding patient features
    return lookup.loc[meta_df["patient_id"].to_numpy(dtype=int)].to_numpy(dtype=np.float32)


In [ ]:
# 5) Base signal tensors and labels
X_train_sig = X_windows[train_mask].astype(np.float32)
X_test_sig = X_windows[test_mask].astype(np.float32)
y_train = y_windows[train_mask].astype(np.int64)
y_test = y_windows[test_mask].astype(np.int64)


In [ ]:
# 6) Additional modalities aligned to windows
X_train_hrv = map_patient_features_to_windows(train_meta, train_patient_df, hrv_cols)
X_test_hrv = map_patient_features_to_windows(test_meta, test_patient_df, hrv_cols)

X_train_demo = map_patient_features_to_windows(train_meta, train_patient_df, demo_cols)
X_test_demo = map_patient_features_to_windows(test_meta, test_patient_df, demo_cols)


In [ ]:
# 7) Three input architectures as dictionaries
nn_input_sets = {
    # ECG signals only, with window-level labels and metadata for potential use in modeling or analysis.
    "signals_only": {
        "train": {
            "signals": X_train_sig,
            "labels": y_train,
            "metadata": train_meta,
        },
        "test": {
            "signals": X_test_sig,
            "labels": y_test,
            "metadata": test_meta,
        },
    },
    # ECG signals + HRV features, allowing models to leverage both raw signal patterns and derived heart rate variability metrics for improved classification performance.
    "signals_plus_hrv": {
        "train": {
            "signals": X_train_sig,
            "hrv": X_train_hrv,
            "labels": y_train,
            "metadata": train_meta,
        },
        "test": {
            "signals": X_test_sig,
            "hrv": X_test_hrv,
            "labels": y_test,
            "metadata": test_meta,
        },
    },
    # ECG signals + demographics for Model 3 two-branch fusion.
    "signals_plus_demographics": {
        "train": {
            "signals": X_train_sig,
            "demographics": X_train_demo,
            "labels": y_train,
            "metadata": train_meta,
        },
        "test": {
            "signals": X_test_sig,
            "demographics": X_test_demo,
            "labels": y_test,
            "metadata": test_meta,
        },
    },
}


In [ ]:
# 8) Keep split metadata for reproducibility
# Build split info dictionary to store all relevant information about the train/test split, feature columns, and imputation/scaling parameters
split_info = {
    "train_patient_ids": train_patient_ids,
    "test_patient_ids": test_patient_ids,
    "hrv_features": hrv_cols,
    "demographic_features": demo_cols,
    "tabular_impute_values": impute_values,
    "tabular_mean": tab_mean,
    "tabular_std": tab_std,
}

print("=" * 90)
print("TRAIN/TEST SPLIT SUMMARY (PATIENT-WISE)")
print("=" * 90)
print(f"Train patients: {len(train_patient_ids)} | Test patients: {len(test_patient_ids)}")
print(f"Train windows : {X_train_sig.shape[0]} | Test windows : {X_test_sig.shape[0]}")
print()

print("Architectures prepared:")
print("1) signals_only")
print(f"   train signals: {nn_input_sets['signals_only']['train']['signals'].shape}, labels: {nn_input_sets['signals_only']['train']['labels'].shape}")
print(f"   test  signals: {nn_input_sets['signals_only']['test']['signals'].shape}, labels: {nn_input_sets['signals_only']['test']['labels'].shape}")

print("2) signals_plus_hrv")
print(f"   train signals: {nn_input_sets['signals_plus_hrv']['train']['signals'].shape}, hrv: {nn_input_sets['signals_plus_hrv']['train']['hrv'].shape}")
print(f"   test  signals: {nn_input_sets['signals_plus_hrv']['test']['signals'].shape}, hrv: {nn_input_sets['signals_plus_hrv']['test']['hrv'].shape}")

print("3) signals_plus_demographics")
print(f"   train signals: {nn_input_sets['signals_plus_demographics']['train']['signals'].shape}, demographics: {nn_input_sets['signals_plus_demographics']['train']['demographics'].shape}")
print(f"   test  signals: {nn_input_sets['signals_plus_demographics']['test']['signals'].shape}, demographics: {nn_input_sets['signals_plus_demographics']['test']['demographics'].shape}")

## Model Definition and Training

Three neural network architectures are developed to demonstrate multimodal learning:

1. **ECG Signals Only**: LSTM network processes 1D temporal ECG windows
2. **Signals + HRV Metrics**: LSTM for signals combined with a dense network for HRV features; outputs concatenated before final classification
3. **Signals + Demographics**: LSTM for signals combined with a dense network for demographics; outputs concatenated before final classification

Each architecture learns a joint representation by fusing modality-specific encoders at an intermediate fusion layer.

In [ ]:
# ------------------------------------------------------------------
# Label Conversion: 1=unhealthy, 0=healthy (clinical significance)
# ------------------------------------------------------------------
# Invert labels so that 1 represents the pathological/unhealthy state
# This reflects clinical intuition where positive diagnosis = abnormality

for arch_name in nn_input_sets.keys():
    nn_input_sets[arch_name]["train"]["labels"] = 1 - nn_input_sets[arch_name]["train"]["labels"]
    nn_input_sets[arch_name]["test"]["labels"] = 1 - nn_input_sets[arch_name]["test"]["labels"]

print("=" * 90)
print("LABEL CONVERSION")
print("=" * 90)
print("Labels inverted: 1 = unhealthy/pathological, 0 = healthy/normal")
print()

# Verify distribution
for arch_name in ["signals_only"]:
    y_train = nn_input_sets[arch_name]["train"]["labels"]
    y_test = nn_input_sets[arch_name]["test"]["labels"]
    
    print(f"{arch_name}:")
    print(f"  Train: {np.sum(y_train == 0)} healthy, {np.sum(y_train == 1)} unhealthy")
    print(f"  Test : {np.sum(y_test == 0)} healthy, {np.sum(y_test == 1)} unhealthy")
    print()

## Model 1: ECG Signals Only (LSTM)

### Network Architecture

**Input (12)**: 500 timesteps of 12-lead ECG data, one neuron per lead.

**Bidirectional LSTM (128 forward + 128 backward)**: Learns temporal patterns in both directions across the 500 timesteps. Each neuron maintains memory of important signal features. Output: 256 features per timestep. For more information about LSTM, please visit [Understanding LSTMs](https://colah.github.io/posts/2015-08-Understanding-LSTMs/).

**Last Hidden-State Concatenation (256)**: Concatenates the final top-layer hidden states from the forward and backward directions to create a fixed-size sequence representation (256 features).

**Dense 256**: Fully-connected layer with ReLU activation to learn non-linear combinations of the hidden-state representation.

**Dense 128**: Compresses from 256 to 128 features, progressively filtering to important diagnostic information.

**Dense 64**: Further compression from 128 to 64 features.

**Output (1 + Sigmoid)**: Single neuron outputs probability [0, 1] where 0 = healthy, 1 = unhealthy.

**Regularization**: BatchNorm, ReLU, and Dropout (0.3) applied after each dense layer to prevent overfitting.

In [ ]:
# ------------------------------------------------------------------
# Model 1: ECG Signals Only
# LSTM network that processes temporal ECG windows
# Input shape: (batch, n_samples, n_leads) = (batch, 500, 12)
# Output: binary classification (healthy=0, unhealthy=1)
# ------------------------------------------------------------------

class ECGSignalsOnlyLSTM(nn.Module):
    """
    LSTM-based network for ECG signal classification.
    
    Architecture:
    - Input: ECG windows (batch, time_steps=500, channels=12)
    - LSTM layers: 2 stacked layers with 128 hidden units each
    - Bidirectional LSTM to capture patterns in both directions
    - Last-layer hidden-state concatenation (forward + backward)
    - Dense layers: 256 -> 128 -> 64 -> 1 (sigmoid for binary classification)
    """
    
    def __init__(self, n_leads=12, lstm_hidden=128, lstm_layers=2, dropout=0.3):
        super(ECGSignalsOnlyLSTM, self).__init__()
        
        self.lstm = nn.LSTM(
            input_size=n_leads,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if lstm_layers > 1 else 0,
        )
        
        # Bidirectional top-layer summary: forward hidden + backward hidden.
        lstm_output_size = 2 * lstm_hidden
        
        self.fc_layers = nn.Sequential(
            nn.Linear(lstm_output_size, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            
            nn.Linear(64, 1),
            nn.Sigmoid(),
        )
    
    # Function which defines the forward pass of the model.
    def forward(self, x):
        """
        Forward pass.
        
        Parameters
        ----------
        x : torch.Tensor
            Shape (batch_size, n_samples, n_leads)
        
        Returns
        -------
        torch.Tensor
            Shape (batch_size, 1) - predicted probability of unhealthy (1)
        """
        # LSTM forward returns per-timestep outputs and final hidden/cell states.
        _, (h_n, _) = self.lstm(x)
        
        # Take top-layer forward/backward hidden states and concatenate them.
        h_last = torch.cat([h_n[-2], h_n[-1]], dim=1)  # (batch, 2*lstm_hidden)
        
        # Classification layers
        logits = self.fc_layers(h_last)
        
        return logits

# Instantiate model on device
model_ecg_only = ECGSignalsOnlyLSTM(
    n_leads=12,
    lstm_hidden=128,
    lstm_layers=2,
    dropout=0.3,
).to(device)

print("=" * 90)
print("MODEL 1: ECG SIGNALS ONLY - LSTM")
print("=" * 90)
print(f"Device: {device}")
print()
print("Architecture:")
print(model_ecg_only)
print()

# Count parameters
n_params_ecg = sum(p.numel() for p in model_ecg_only.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params_ecg:,}")
print()

### Training Setup

#### Custom PyTorch Dataset & DataLoaders

In [ ]:
# ------------------------------------------------------------------
# Custom Dataset Class for ECG Windows
# ------------------------------------------------------------------

class ECGWindowDataset(Dataset):
    """PyTorch Dataset for ECG signal windows"""
    
    def __init__(self, signals, labels):
        """
        Parameters
        ----------
        signals : np.ndarray
            Shape (n_samples, 500, 12) - ECG windows
        labels : np.ndarray
            Shape (n_samples,) - binary labels
        """
        self.signals = torch.FloatTensor(signals)
        self.labels = torch.FloatTensor(labels).unsqueeze(1)  # Make 2D: (n,1)
    
    def __len__(self):
        return len(self.signals)
    
    def __getitem__(self, idx):
        return self.signals[idx], self.labels[idx]

# Create datasets from the train/test split
train_signals = nn_input_sets["signals_only"]["train"]["signals"]
train_labels = nn_input_sets["signals_only"]["train"]["labels"]

test_signals = nn_input_sets["signals_only"]["test"]["signals"]
test_labels = nn_input_sets["signals_only"]["test"]["labels"]

# Create PyTorch datasets
train_dataset = ECGWindowDataset(train_signals, train_labels)
test_dataset = ECGWindowDataset(test_signals, test_labels)

# Create DataLoaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

print("=" * 90)
print("DATA LOADERS CREATED")
print("=" * 90)
print(f"Train samples: {len(train_dataset)} in {len(train_loader)} batches of {batch_size}")
print(f"Test samples: {len(test_dataset)} in {len(test_loader)} batches of {batch_size}")
print()

#### Training Configuration & Functions

Binary Cross Entropy loss:

$\mathcal{L} = -\left[y\log(p) + (1-y)\log(1-p)\right]$

In [ ]:
# ------------------------------------------------------------------
# Training Configuration
# ------------------------------------------------------------------

learning_rate = 0.001
num_epochs = 100
patience = 15  # Early stopping: stop if validation doesn't improve for 15 epochs

# Loss function & Optimizer
criterion = nn.BCELoss()  # Binary Cross Entropy (output is already sigmoid)
optimizer = optim.Adam(model_ecg_only.parameters(), lr=learning_rate)

# Learning rate scheduler (reduce LR if validation loss plateaus)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    mode='min', 
    factor=0.5, 
    patience=5
)

print("=" * 90)
print("TRAINING CONFIGURATION")
print("=" * 90)
print(f"Loss function: {criterion.__class__.__name__}")
print(f"Optimizer: {optimizer.__class__.__name__}")
print(f"Learning rate: {learning_rate}")
print(f"Max epochs: {num_epochs}")
print(f"Early stopping patience: {patience} epochs")
print(f"Device: {device}")
print()

# ------------------------------------------------------------------
# Training & Validation Functions
# ------------------------------------------------------------------

def train_epoch(model, train_loader, criterion, optimizer, device):
    """Train for one epoch"""
    model.train()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    
    for batch_signals, batch_labels in train_loader:
        # Send batch to device (signals + labels)
        batch_signals = batch_signals.to(device)
        batch_labels = batch_labels.to(device)
        
        # Forward pass
        # Produce probabilities with shape (batch_size, 1) where values are in [0,1] due to sigmoid activation in the model's last layer
        outputs = model(batch_signals)
        loss = criterion(outputs, batch_labels)
        
        # Backward pass
        # Clear gradients, compute new gradients via backpropagation, and update weights
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Accumulate metrics
        # Store the loss for this batch and detach predictions and labels to move them to CPU for metric computation after the epoch
        total_loss += loss.item()
        all_preds.append(outputs.detach().cpu().numpy())
        all_labels.append(batch_labels.cpu().numpy())
    
    # Compute epoch metrics
    all_preds = np.concatenate(all_preds, axis=0).squeeze()
    all_labels = np.concatenate(all_labels, axis=0).squeeze()
    
    avg_loss = total_loss / len(train_loader)
    accuracy = accuracy_score(all_labels.round(), all_preds.round())
    
    return avg_loss, accuracy

def validate(model, test_loader, criterion, device):
    """Validate on test set"""
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    
    # Disable gradient computation for validation to speed up inference and reduce memory usage
    with torch.no_grad():
        for batch_signals, batch_labels in test_loader:
            batch_signals = batch_signals.to(device)
            batch_labels = batch_labels.to(device)
            
            outputs = model(batch_signals)
            loss = criterion(outputs, batch_labels)

            # Compute loss on validation batch and store predictions and labels for metric computation after the epoch
            total_loss += loss.item()
            all_preds.append(outputs.cpu().numpy())
            all_labels.append(batch_labels.cpu().numpy())
    
    # Compute test metrics
    all_preds = np.concatenate(all_preds, axis=0).squeeze()
    all_labels = np.concatenate(all_labels, axis=0).squeeze()
    
    avg_loss = total_loss / len(test_loader)
    accuracy = accuracy_score(all_labels.round(), all_preds.round())
    auc = roc_auc_score(all_labels, all_preds)
    
    return avg_loss, accuracy, auc, all_preds, all_labels

print("Training functions defined.")
print()

#### Training Loop - Model 1 (ECG Signals Only)

In [ ]:
# ------------------------------------------------------------------
# Training Loop with Early Stopping
# ------------------------------------------------------------------

# Reset model before training
# Define it and send to device again to ensure we start with fresh weights, especially if this cell is re-run after previous training attempts
model_ecg_only = ECGSignalsOnlyLSTM(
    n_leads=12,
    lstm_hidden=128,
    lstm_layers=2,
    dropout=0.3,
).to(device)

# Define optimizer and scheduler again to reset their states as well
optimizer = optim.Adam(model_ecg_only.parameters(), lr=learning_rate)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=5,
)

# Storage for metrics
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
    'val_auc': [],
}

best_val_loss = float('inf')
best_val_auc = 0.0
epochs_without_improvement = 0
best_epoch = 0
best_model_state = None

print("=" * 90)
print("TRAINING MODEL 1: ECG SIGNALS ONLY LSTM")
print("=" * 90)
print()

model1_ckpt_name = "model1_ecg_signals_only_hiddenstate_best.pt"
model1_ckpt_path = os.path.join(MODELS_DIR, model1_ckpt_name)
num_epochs_to_run = num_epochs
if os.path.exists(model1_ckpt_path):
    checkpoint_metadata = load_model_checkpoint(model_ecg_only, filename=model1_ckpt_name, device=device)
    print(f"Found existing checkpoint '{model1_ckpt_name}'. Skipping training loop.")
    best_model_state = {k: v.detach().cpu().clone() for k, v in model_ecg_only.state_dict().items()}
    num_epochs_to_run = 0
    if isinstance(checkpoint_metadata, dict):
        best_epoch = int(checkpoint_metadata.get("best_epoch", best_epoch))
        best_val_auc = float(checkpoint_metadata.get("best_val_auc", best_val_auc))
        best_val_loss = float(checkpoint_metadata.get("best_val_loss", best_val_loss))

# Training loop (it does not run if a valid checkpoint was loaded)
for epoch in range(num_epochs_to_run):
    # Train epoch
    train_loss, train_acc = train_epoch(model_ecg_only, train_loader, criterion, optimizer, device)
    
    # Validate
    val_loss, val_acc, val_auc, val_preds, val_labels = validate(model_ecg_only, test_loader, criterion, device)
    
    # Store metrics
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_auc'].append(val_auc)
    
    # Update learning rate scheduler
    scheduler.step(val_loss)
    
    # Early stopping & model checkpointing
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_val_loss = val_loss
        best_epoch = epoch + 1
        epochs_without_improvement = 0
        best_model_state = {k: v.detach().cpu().clone() for k, v in model_ecg_only.state_dict().items()}
        print(f"Epoch {epoch+1:3d} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val AUC: {val_auc:.4f} STAR (BEST)")
    else:
        epochs_without_improvement += 1
        print(f"Epoch {epoch+1:3d} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val AUC: {val_auc:.4f}")
    
    # Early stopping
    if epochs_without_improvement >= patience:
        print(f"\nEarly stopping at epoch {epoch+1}. No improvement for {patience} epochs.")
        break

if best_model_state is None:
    raise RuntimeError("No best model state found. Training did not produce a valid checkpoint.")

# Restore best model when training is complete or if a checkpoint was loaded at the start
model_ecg_only.load_state_dict(best_model_state)

# Persist best model checkpoint to disk so retraining is not required.
saved_model_path = save_model_checkpoint(
    model_ecg_only,
    filename=model1_ckpt_name,
    metadata={
        "model_name": "ECGSignalsOnlyLSTM",
        "best_epoch": int(best_epoch),
        "best_val_auc": float(best_val_auc),
        "best_val_loss": float(best_val_loss),
        "sampling_rate_hz": int(sampling_rate_to_plot),
        "window_sec": float(WINDOW_SEC),
        "step_sec": float(STEP_SEC),
    },
)

print()
print("=" * 90)
print("TRAINING COMPLETE")
print("=" * 90)
print(f"Best epoch: {best_epoch}")
print(f"Best Validation Loss: {best_val_loss:.4f}")
print(f"Best Validation AUC: {best_val_auc:.4f}")
print(f"Total epochs trained: {len(history['train_loss'])}")
print(f"Saved best model to: {saved_model_path}")
print()

### Training History & Performance Visualization

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Loss curve
axes[0].plot(history['train_loss'], label='Train Loss', linewidth=2, color='steelblue')
axes[0].plot(history['val_loss'], label='Val Loss', linewidth=2, color='coral')
axes[0].set_xlabel('Epoch', fontsize=11, fontweight='bold')
axes[0].set_ylabel('BCE Loss', fontsize=11, fontweight='bold')
axes[0].set_title('Training & Validation Loss', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

# Accuracy curve
axes[1].plot(history['train_acc'], label='Train Accuracy', linewidth=2, color='steelblue')
axes[1].plot(history['val_acc'], label='Val Accuracy', linewidth=2, color='coral')
axes[1].set_xlabel('Epoch', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Accuracy', fontsize=11, fontweight='bold')
axes[1].set_title('Training & Validation Accuracy', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)

# AUC curve
axes[2].plot(history['val_auc'], label='Val ROC-AUC', linewidth=2, color='mediumseagreen')
axes[2].set_xlabel('Epoch', fontsize=11, fontweight='bold')
axes[2].set_ylabel('ROC-AUC', fontsize=11, fontweight='bold')
axes[2].set_title('Validation ROC-AUC', fontsize=12, fontweight='bold')
axes[2].legend(fontsize=10)
axes[2].grid(alpha=0.3)
axes[2].set_ylim([0, 1.05])

plt.tight_layout()
save_figure(fig, "model1_training_history")
plt.show()

print("Training history plots saved.")
print()

### Final Model Evaluation on Test Set

In [ ]:
# Evaluate on test set with best model checkpoint.
# This allows running the notebook from this section onward without retraining.
model_ecg_only = ECGSignalsOnlyLSTM(
    n_leads=12,
    lstm_hidden=128,
    lstm_layers=2,
    dropout=0.3,
).to(device)

checkpoint_metadata = load_model_checkpoint(
    model_ecg_only,
    filename="model1_ecg_signals_only_hiddenstate_best.pt",
    device=device,
)

if checkpoint_metadata is not None:
    print("Loaded checkpoint metadata:")
    print(checkpoint_metadata)
    print()

_, test_acc, test_auc, test_preds, test_labels = validate(model_ecg_only, test_loader, criterion, device)

# Round predictions for classification metrics
# We round the predicted probabilities to get binary class predictions where values >= 0.5 are classified as 1 (unhealthy) and values < 0.5 are classified as 0 (healthy)
# It is important to say that this threshold can be adjusted based on the desired sensitivity/specificity trade-off, but 0.5 is a common default for balanced datasets and binary classification tasks
test_preds_binary = test_preds.round()
test_labels_binary = test_labels.round()

# Compute metrics
cm = confusion_matrix(test_labels_binary, test_preds_binary)
tn, fp, fn, tp = cm.ravel()

sensitivity = tp / (tp + fn)  # True Positive Rate
specificity = tn / (tn + fp)  # True Negative Rate
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
f1 = f1_score(test_labels_binary, test_preds_binary)

print("=" * 90)
print("TEST SET EVALUATION - MODEL 1 (ECG SIGNALS ONLY)")
print("=" * 90)
print(f"Accuracy:    {test_acc:.4f}")
print(f"ROC-AUC:     {test_auc:.4f}")
print(f"F1-Score:    {f1:.4f}")
print(f"Sensitivity: {sensitivity:.4f} (True Positive Rate)")
print(f"Specificity: {specificity:.4f} (True Negative Rate)")
print(f"Precision:   {precision:.4f}")
print()

print("Confusion Matrix:")
print(f"  {' ':15} Predicted Healthy  Predicted Unhealthy")
print(f"  Actual Healthy        {tn:4d}             {fp:4d}")
print(f"  Actual Unhealthy      {fn:4d}             {tp:4d}")
print()

print("Classification Report:")
print(classification_report(test_labels_binary, test_preds_binary, 
                          target_names=['Healthy', 'Unhealthy'],
                          digits=4))
print()

### Normalized Confusion Matrix (Test Set, %)

In [ ]:
# Row-normalized confusion matrix in percentage (rows = true class).

# Recompute predictions to keep this cell self-contained.
_, _, _, test_preds, test_labels = validate(model_ecg_only, test_loader, criterion, device)

test_preds_binary = np.rint(test_preds).astype(int)
test_labels_binary = np.rint(test_labels).astype(int)

labels_order = [0, 1]  # 0 = Healthy, 1 = Unhealthy
class_names = ["Healthy", "Unhealthy"]

cm_counts = confusion_matrix(test_labels_binary, test_preds_binary, labels=labels_order)
cm_percent = confusion_matrix(
    test_labels_binary,
    test_preds_binary,
    labels=labels_order,
    normalize="true",
) * 100.0

# Keep plotting style local to this figure to avoid affecting other notebook plots.
with sns.plotting_context(
    "notebook",
    rc={
        "font.size": 12,
        "axes.titlesize": 16,
        "axes.labelsize": 13,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
    },
):
    fig, ax = plt.subplots(figsize=(8.2, 7.2))
    heat = sns.heatmap(
        cm_percent,
        annot=False,
        fmt=".1f",
        cmap="Blues",
        vmin=0,
        vmax=100,
        cbar=True,
        cbar_kws={
            "label": "Row-normalized percentage [%]",
            "shrink": 0.65,
            "pad": 0.15,
        },
        xticklabels=class_names,
        yticklabels=class_names,
        linewidths=1.6,
        linecolor="white",
        square=True,
        ax=ax,
    )

# Improve axis readability.
ax.set_xlabel("Predicted label", fontweight="bold", labelpad=10)
ax.set_ylabel("True label", fontweight="bold", labelpad=10)
ax.set_title("Normalized Confusion Matrix (%) - Model 1", fontweight="bold", pad=14)
ax.set_xticklabels(class_names, rotation=0, ha="center")
ax.set_yticklabels(class_names, rotation=0, va="center")

# Make the colorbar labels readable and consistent with axis labels.
cbar = heat.collections[0].colorbar
cbar.ax.tick_params(labelsize=11)
cbar.set_label("Row-normalized percentage [%]", fontsize=12)

# Add percentage + absolute count in each cell with contrast-aware text color.
threshold = 60.0
for i in range(cm_percent.shape[0]):
    for j in range(cm_percent.shape[1]):
        pct = cm_percent[i, j]
        cnt = cm_counts[i, j]
        text_color = "white" if pct >= threshold else "#1f2937"
        ax.text(
            j + 0.5,
            i + 0.5,
            f"{pct:.1f}%\n(n={cnt})",
            ha="center",
            va="center",
            fontsize=12,
            fontweight="bold",
            color=text_color,
        )

# Add row support counts at the right side for didactic clarity.
row_totals = cm_counts.sum(axis=1)
for i, total in enumerate(row_totals):
    ax.text(
        cm_percent.shape[1] + 0.03,
        i + 0.5,
        f"Total={total}",
        transform=ax.transData,
        ha="left",
        va="center",
        fontsize=11,
        color="#374151",
        clip_on=False,
    )

plt.tight_layout()
save_figure(fig, "model1_test_confusion_matrix_normalized_percentage")
plt.show()

cm_percent_df = pd.DataFrame(cm_percent, index=class_names, columns=class_names)
print("Normalized confusion matrix (%):")
display(cm_percent_df.round(2))


### ROC Curve (Test Set)

In [ ]:
# Recompute probabilities on test set to ensure this cell is self-contained
_, _, _, test_preds, test_labels = validate(model_ecg_only, test_loader, criterion, device)

fpr, tpr, _ = roc_curve(test_labels, test_preds)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(
    fpr,
    tpr,
    color="royalblue",
    linewidth=2.5,
    label=f"ROC curve (AUC = {roc_auc:.4f})",
)
ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    color="gray",
    linewidth=1.5,
    label="Random classifier",
)

ax.set_xlim([-0.01, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel("False Positive Rate", fontsize=11, fontweight="bold")
ax.set_ylabel("True Positive Rate", fontsize=11, fontweight="bold")
ax.set_title("ROC Curve - Model 1 (ECG Signals Only)", fontsize=12, fontweight="bold")
ax.grid(alpha=0.3)
ax.legend(loc="lower right", fontsize=10)

plt.tight_layout()
save_figure(fig, "model1_test_roc_curve")
plt.show()

print(f"ROC-AUC (test): {roc_auc:.4f}")

## Model 2: ECG Signals + HRV Metrics (Two-Branch Fusion)
In this model we fuse:
- an **LSTM encoder** for the ECG window
- an **MLP encoder** for the HRV feature vector

The two embeddings are concatenated and passed to a small classification head.

### Dataset & DataLoaders (ECG + HRV)
This model uses a **two-branch** input:
- **Branch 1**: ECG signal windows (same as Model 1)
- **Branch 2**: HRV feature vectors mapped to each window

In this step we prepare the PyTorch dataset and dataloaders (no model definition yet).

In [ ]:
# ------------------------------------------------------------------
# Custom Dataset Class for ECG Windows + HRV features (Model 2)
# ------------------------------------------------------------------

class ECGWindowHRVDataset(Dataset):
    """PyTorch Dataset for (ECG window, HRV vector) pairs."""

    def __init__(self, signals, hrv_features, labels):
        """
        Parameters
        ----------
        signals : np.ndarray
            Shape (n_samples, 500, 12) - ECG windows
        hrv_features : np.ndarray
            Shape (n_samples, n_hrv_features) - HRV feature vector per window
        labels : np.ndarray
            Shape (n_samples,) - binary labels (0/1)
        """
        if len(signals) != len(hrv_features) or len(signals) != len(labels):
            raise ValueError(
                "signals, hrv_features, and labels must have the same first dimension"
            )

        self.signals = torch.as_tensor(signals, dtype=torch.float32)
        self.hrv = torch.as_tensor(hrv_features, dtype=torch.float32)
        self.labels = torch.as_tensor(labels, dtype=torch.float32).unsqueeze(1)  # (n, 1)

    def __len__(self):
        return self.signals.shape[0]

    def __getitem__(self, idx):
        return self.signals[idx], self.hrv[idx], self.labels[idx]

# ------------------------------------------------------------------
# Create datasets + dataloaders for Model 2 from the prepared input set
# ------------------------------------------------------------------

# Safe default if this cell is executed out of order.
try:
    batch_size
except NameError:
    batch_size = 32

train_signals_m2 = nn_input_sets["signals_plus_hrv"]["train"]["signals"]
train_hrv_m2 = nn_input_sets["signals_plus_hrv"]["train"]["hrv"]
train_labels_m2 = nn_input_sets["signals_plus_hrv"]["train"]["labels"]

test_signals_m2 = nn_input_sets["signals_plus_hrv"]["test"]["signals"]
test_hrv_m2 = nn_input_sets["signals_plus_hrv"]["test"]["hrv"]
test_labels_m2 = nn_input_sets["signals_plus_hrv"]["test"]["labels"]

train_dataset_m2 = ECGWindowHRVDataset(train_signals_m2, train_hrv_m2, train_labels_m2)
test_dataset_m2 = ECGWindowHRVDataset(test_signals_m2, test_hrv_m2, test_labels_m2)

# Keep separate loader names so Model 1 training cells remain unchanged.
train_loader_m2 = DataLoader(train_dataset_m2, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader_m2 = DataLoader(test_dataset_m2, batch_size=batch_size, shuffle=False, num_workers=0)

print("=" * 90)
print("DATA LOADERS CREATED - MODEL 2 (ECG + HRV)")
print("=" * 90)
print(f"Train samples: {len(train_dataset_m2)} in {len(train_loader_m2)} batches of {batch_size}")
print(f"Test samples:  {len(test_dataset_m2)} in {len(test_loader_m2)} batches of {batch_size}")
print()
print(f"ECG window shape: {train_signals_m2.shape[1:]} (samples, leads)")
print(f"HRV vector shape: {train_hrv_m2.shape[1:]} (n_hrv_features,)")
print()

# Quick sanity check on one batch (shapes only).
batch_sig, batch_hrv, batch_lab = next(iter(train_loader_m2))
print("One batch shapes:")
print(f"  signals: {tuple(batch_sig.shape)}")
print(f"  hrv:     {tuple(batch_hrv.shape)}")
print(f"  labels:  {tuple(batch_lab.shape)}")

### Model Definition (Two-Branch Fusion Network)
We define a fusion model with:
- an LSTM encoder for the ECG window
- an MLP encoder for the HRV feature vector
- a small classification head on the concatenated embeddings.

In [ ]:
# ------------------------------------------------------------------
# Model 2: Two-branch fusion network (ECG LSTM + HRV MLP)
# ------------------------------------------------------------------

class ECGSignalsHRVFusionNet(nn.Module):
    """
    Two-branch network for ECG window classification with HRV features.

    Inputs
    ------
    signals : (batch, time_steps=500, n_leads=12)
    hrv     : (batch, n_hrv_features)

    Output
    ------
    prob_unhealthy : (batch, 1) in [0, 1]
    """

    def __init__(
        self,
        n_leads=12,
        n_hrv_features=7,
        lstm_hidden=128,
        lstm_layers=2,
        dropout=0.3,
        ecg_embed_dim=64,
        hrv_embed_dim=16,
    ):
        super().__init__()

        # ECG branch: use top-layer bidirectional hidden-state summary.
        self.ecg_lstm = nn.LSTM(
            input_size=n_leads,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if lstm_layers > 1 else 0.0,
        )
        self.ecg_proj = nn.Sequential(
            nn.Linear(2 * lstm_hidden, ecg_embed_dim),
            nn.BatchNorm1d(ecg_embed_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        # HRV branch (small MLP for low-dimensional tabular features).
        self.hrv_mlp = nn.Sequential(
            nn.Linear(n_hrv_features, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, hrv_embed_dim),
            nn.BatchNorm1d(hrv_embed_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
        )

        fusion_dim = ecg_embed_dim + hrv_embed_dim
        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
            nn.Sigmoid(),
        )

    def forward(self, signals, hrv):
        _, (h_n, _) = self.ecg_lstm(signals)
        ecg_summary = torch.cat([h_n[-2], h_n[-1]], dim=1)  # (batch, 2*lstm_hidden)
        ecg_embed = self.ecg_proj(ecg_summary)
        hrv_embed = self.hrv_mlp(hrv)
        fused = torch.cat([ecg_embed, hrv_embed], dim=1)
        return self.classifier(fused)

n_hrv_features = int(nn_input_sets["signals_plus_hrv"]["train"]["hrv"].shape[1])
model_ecg_hrv = ECGSignalsHRVFusionNet(
    n_leads=12,
    n_hrv_features=n_hrv_features,
    lstm_hidden=128,
    lstm_layers=2,
    dropout=0.3,
    ecg_embed_dim=64,
    hrv_embed_dim=16,
).to(device)

print("=" * 90)
print("MODEL 2: ECG + HRV (FUSION)")
print("=" * 90)
print(f"Device: {device}")
print(f"HRV features: {n_hrv_features}")
print()
print(model_ecg_hrv)
print()
n_params_m2 = sum(p.numel() for p in model_ecg_hrv.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params_m2:,}")

### Training (ECG + HRV)
We train Model 2 using the same approach as Model 1:
- binary cross-entropy loss
- early stopping based on validation ROC AUC
- checkpoint saving of the best model in `../models`.

In [ ]:
# ------------------------------------------------------------------
# Training config + train/validate functions for Model 2
# ------------------------------------------------------------------

learning_rate_m2 = 0.001
num_epochs_m2 = 100
patience_m2 = 15

criterion_m2 = nn.BCELoss()
optimizer_m2 = optim.Adam(model_ecg_hrv.parameters(), lr=learning_rate_m2)
scheduler_m2 = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_m2,
    mode="min",
    factor=0.5,
    patience=5,
)

print("=" * 90)
print("MODEL 2 TRAINING CONFIGURATION")
print("=" * 90)
print(f"Loss function: {criterion_m2.__class__.__name__}")
print(f"Optimizer: {optimizer_m2.__class__.__name__}")
print(f"Learning rate: {learning_rate_m2}")
print(f"Max epochs: {num_epochs_m2}")
print(f"Early stopping patience: {patience_m2} epochs")
print()

def _ensure_column_vector(x: torch.Tensor) -> torch.Tensor:
    # Make sure labels have shape (batch, 1) to match model outputs.
    if x.ndim == 1:
        return x.view(-1, 1)
    if x.ndim == 2 and x.shape[1] == 1:
        return x
    return x.view(-1, 1)

def train_epoch_m2(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    all_probs = []
    all_labels = []

    for batch_signals, batch_hrv, batch_labels in train_loader:
        batch_signals = batch_signals.to(device)
        batch_hrv = batch_hrv.to(device)
        batch_labels = _ensure_column_vector(batch_labels.to(device).float())

        probs = model(batch_signals, batch_hrv)
        loss = criterion(probs, batch_labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += float(loss.item())
        all_probs.append(probs.detach().cpu().numpy())
        all_labels.append(batch_labels.detach().cpu().numpy())

    all_probs = np.concatenate(all_probs, axis=0).reshape(-1)
    all_labels = np.concatenate(all_labels, axis=0).reshape(-1).astype(int)
    avg_loss = total_loss / max(1, len(train_loader))
    acc = accuracy_score(all_labels, (all_probs >= 0.5).astype(int))
    return avg_loss, acc

def validate_m2(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for batch_signals, batch_hrv, batch_labels in loader:
            batch_signals = batch_signals.to(device)
            batch_hrv = batch_hrv.to(device)
            batch_labels = _ensure_column_vector(batch_labels.to(device).float())

            probs = model(batch_signals, batch_hrv)
            loss = criterion(probs, batch_labels)

            total_loss += float(loss.item())
            all_probs.append(probs.detach().cpu().numpy())
            all_labels.append(batch_labels.detach().cpu().numpy())

    all_probs = np.concatenate(all_probs, axis=0).reshape(-1)
    all_labels = np.concatenate(all_labels, axis=0).reshape(-1).astype(int)
    avg_loss = total_loss / max(1, len(loader))
    acc = accuracy_score(all_labels, (all_probs >= 0.5).astype(int))
    auc_val = roc_auc_score(all_labels, all_probs)
    return avg_loss, acc, auc_val, all_probs, all_labels

#### Training Loop - Model 2 (ECG + HRV)
The following cell runs the training loop, restores the best validation model, and saves a checkpoint.

In [ ]:
# ------------------------------------------------------------------
# Training loop for Model 2 + checkpoint saving
# ------------------------------------------------------------------

# Reset model before training (keeps Model 2 runs reproducible).
model_ecg_hrv = ECGSignalsHRVFusionNet(
    n_leads=12,
    n_hrv_features=n_hrv_features,
    lstm_hidden=128,
    lstm_layers=2,
    dropout=0.3,
    ecg_embed_dim=64,
    hrv_embed_dim=16,
).to(device)

optimizer_m2 = optim.Adam(model_ecg_hrv.parameters(), lr=learning_rate_m2)
scheduler_m2 = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_m2,
    mode="min",
    factor=0.5,
    patience=5,
)

history_m2 = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
    "val_auc": [],
}

best_val_auc_m2 = 0.0
best_val_loss_m2 = float("inf")
best_epoch_m2 = 0
epochs_without_improvement_m2 = 0
best_model_state_m2 = None

print("=" * 90)
print("TRAINING MODEL 2: ECG + HRV (FUSION)")
print("=" * 90)
print()

model2_ckpt_name = "model2_ecg_signals_plus_hrv_hiddenstate_best.pt"
model2_ckpt_path = os.path.join(MODELS_DIR, model2_ckpt_name)
num_epochs_to_run_m2 = num_epochs_m2
if os.path.exists(model2_ckpt_path):
    checkpoint_metadata_m2 = load_model_checkpoint(model_ecg_hrv, filename=model2_ckpt_name, device=device)
    print(f"Found existing checkpoint '{model2_ckpt_name}'. Skipping training loop.")
    best_model_state_m2 = {k: v.detach().cpu().clone() for k, v in model_ecg_hrv.state_dict().items()}
    num_epochs_to_run_m2 = 0
    if isinstance(checkpoint_metadata_m2, dict):
        best_epoch_m2 = int(checkpoint_metadata_m2.get("best_epoch", best_epoch_m2))
        best_val_auc_m2 = float(checkpoint_metadata_m2.get("best_val_auc", best_val_auc_m2))
        best_val_loss_m2 = float(checkpoint_metadata_m2.get("best_val_loss", best_val_loss_m2))

for epoch in range(num_epochs_to_run_m2):
    train_loss, train_acc = train_epoch_m2(
        model_ecg_hrv, train_loader_m2, criterion_m2, optimizer_m2, device
    )
    val_loss, val_acc, val_auc, _, _ = validate_m2(
        model_ecg_hrv, test_loader_m2, criterion_m2, device
    )

    history_m2["train_loss"].append(train_loss)
    history_m2["train_acc"].append(train_acc)
    history_m2["val_loss"].append(val_loss)
    history_m2["val_acc"].append(val_acc)
    history_m2["val_auc"].append(val_auc)

    scheduler_m2.step(val_loss)

    if val_auc > best_val_auc_m2:
        best_val_auc_m2 = float(val_auc)
        best_val_loss_m2 = float(val_loss)
        best_epoch_m2 = int(epoch + 1)
        epochs_without_improvement_m2 = 0
        best_model_state_m2 = {
            k: v.detach().cpu().clone() for k, v in model_ecg_hrv.state_dict().items()
        }
        print(
            f"Epoch {epoch+1:3d} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val AUC: {val_auc:.4f} STAR (BEST)"
        )
    else:
        epochs_without_improvement_m2 += 1
        print(
            f"Epoch {epoch+1:3d} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val AUC: {val_auc:.4f}"
        )

    if epochs_without_improvement_m2 >= patience_m2:
        print(f"\nEarly stopping at epoch {epoch+1}. No improvement for {patience_m2} epochs.")
        break

if best_model_state_m2 is None:
    raise RuntimeError("No best model state found for Model 2.")

# Restore + save best checkpoint
model_ecg_hrv.load_state_dict(best_model_state_m2)

saved_model2_path = save_model_checkpoint(
    model_ecg_hrv,
    filename=model2_ckpt_name,
    metadata={
        "model_name": "ECGSignalsHRVFusionNet",
        "best_epoch": int(best_epoch_m2),
        "best_val_auc": float(best_val_auc_m2),
        "best_val_loss": float(best_val_loss_m2),
        "sampling_rate_hz": int(sampling_rate_to_plot),
        "window_sec": float(WINDOW_SEC),
        "step_sec": float(STEP_SEC),
        "n_hrv_features": int(n_hrv_features),
    },
)

print()
print("=" * 90)
print("MODEL 2 TRAINING COMPLETE")
print("=" * 90)
print(f"Best epoch: {best_epoch_m2}")
print(f"Best Validation Loss: {best_val_loss_m2:.4f}")
print(f"Best Validation AUC: {best_val_auc_m2:.4f}")
print(f"Total epochs trained: {len(history_m2['train_loss'])}")
print(f"Saved best model to: {saved_model2_path}")

### Training History & Performance Visualization (Model 2)

In [ ]:
# Plot training history for Model 2
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Loss curve
axes[0].plot(history_m2['train_loss'], label='Train Loss', linewidth=2, color='steelblue')
axes[0].plot(history_m2['val_loss'], label='Val Loss', linewidth=2, color='coral')
axes[0].set_xlabel('Epoch', fontsize=11, fontweight='bold')
axes[0].set_ylabel('BCE Loss', fontsize=11, fontweight='bold')
axes[0].set_title('Training & Validation Loss', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

# Accuracy curve
axes[1].plot(history_m2['train_acc'], label='Train Accuracy', linewidth=2, color='steelblue')
axes[1].plot(history_m2['val_acc'], label='Val Accuracy', linewidth=2, color='coral')
axes[1].set_xlabel('Epoch', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Accuracy', fontsize=11, fontweight='bold')
axes[1].set_title('Training & Validation Accuracy', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)

# AUC curve
axes[2].plot(history_m2['val_auc'], label='Val ROC-AUC', linewidth=2, color='mediumseagreen')
axes[2].set_xlabel('Epoch', fontsize=11, fontweight='bold')
axes[2].set_ylabel('ROC-AUC', fontsize=11, fontweight='bold')
axes[2].set_title('Validation ROC-AUC', fontsize=12, fontweight='bold')
axes[2].legend(fontsize=10)
axes[2].grid(alpha=0.3)
axes[2].set_ylim([0, 1.05])

plt.tight_layout()
save_figure(fig, "model2_training_history")
plt.show()

print("Model 2 training history plots saved.")
print()

### Final Model Evaluation on Test Set (Model 2)
This section loads the best saved checkpoint and evaluates on the test set.
It reports standard classification metrics and produces:
- a row-normalized confusion matrix (percentages)
- the ROC curve with AUC.

In [ ]:
# ------------------------------------------------------------------
# Final Model 2 evaluation on test set (load best checkpoint)
# ------------------------------------------------------------------

# Re-instantiate and load the saved best checkpoint (so this cell can run independently).
model2_ckpt_name = "model2_ecg_signals_plus_hrv_hiddenstate_best.pt"
model_ecg_hrv_eval = ECGSignalsHRVFusionNet(
    n_leads=12,
    n_hrv_features=n_hrv_features,
    lstm_hidden=128,
    lstm_layers=2,
    dropout=0.3,
    ecg_embed_dim=64,
    hrv_embed_dim=16,
).to(device)

_ = load_model_checkpoint(model_ecg_hrv_eval, filename=model2_ckpt_name, device=device)
model_ecg_hrv_eval.eval()

# Collect predictions
y_true_m2 = []
y_prob_m2 = []
y_pred_m2 = []

with torch.no_grad():
    for signals, hrv, labels in test_loader_m2:
        signals = signals.to(device)
        hrv = hrv.to(device)
        labels = labels.to(device).float().view(-1, 1)

        probs = model_ecg_hrv_eval(signals, hrv)
        preds = (probs >= 0.5).float()

        y_true_m2.append(labels.detach().cpu().numpy())
        y_prob_m2.append(probs.detach().cpu().numpy())
        y_pred_m2.append(preds.detach().cpu().numpy())

y_true_m2 = np.concatenate(y_true_m2, axis=0).reshape(-1).astype(int)
y_pred_m2 = np.concatenate(y_pred_m2, axis=0).reshape(-1).astype(int)
y_prob_m2 = np.concatenate(y_prob_m2, axis=0).reshape(-1).astype(float)

# Metrics
acc_m2 = accuracy_score(y_true_m2, y_pred_m2)
auc_m2 = roc_auc_score(y_true_m2, y_prob_m2)
print("=" * 90)
print("MODEL 2: FINAL EVALUATION (TEST SET)")
print("=" * 90)
print(f"Accuracy: {acc_m2:.4f}")
print(f"ROC AUC:  {auc_m2:.4f}")
print()
print("Classification report:")
print(classification_report(y_true_m2, y_pred_m2, digits=4))

# Confusion matrix: counts + row-normalized percentages
cm_counts_m2 = confusion_matrix(y_true_m2, y_pred_m2, labels=[0, 1])
cm_percent_m2 = confusion_matrix(y_true_m2, y_pred_m2, labels=[0, 1], normalize="true") * 100.0
class_names = ["Healthy (0)", "Not Healthy (1)"]

fig, ax = plt.subplots(figsize=(8.8, 7.0))
with sns.axes_style("white"):
    heat = sns.heatmap(
        cm_percent_m2,
        annot=False,
        fmt=".1f",
        cmap="Blues",
        vmin=0,
        vmax=100,
        cbar=True,
        cbar_kws={
            "label": "Row-normalized percentage [%]",
            "shrink": 0.65,
            "pad": 0.15,
        },
        xticklabels=class_names,
        yticklabels=class_names,
        linewidths=1.6,
        linecolor="white",
        square=True,
        ax=ax,
    )

# Improve axis readability.
ax.set_xlabel("Predicted label", fontweight="bold", labelpad=10)
ax.set_ylabel("True label", fontweight="bold", labelpad=10)
ax.set_title("Normalized Confusion Matrix (%) - Model 2", fontweight="bold", pad=14)
ax.set_xticklabels(class_names, rotation=0, ha="center")
ax.set_yticklabels(class_names, rotation=0, va="center")

# Make the colorbar labels readable and consistent with axis labels.
cbar = heat.collections[0].colorbar
cbar.ax.tick_params(labelsize=11)
cbar.set_label("Row-normalized percentage [%]", fontsize=12)

# Add percentage + absolute count in each cell with contrast-aware text color.
threshold = 60.0
for i in range(cm_percent_m2.shape[0]):
    for j in range(cm_percent_m2.shape[1]):
        pct = cm_percent_m2[i, j]
        cnt = cm_counts_m2[i, j]
        text_color = "white" if pct >= threshold else "#1f2937"
        ax.text(
            j + 0.5,
            i + 0.5,
            f"{pct:.1f}%\n(n={cnt})",
            ha="center",
            va="center",
            fontsize=12,
            fontweight="bold",
            color=text_color,
        )

# Add row support counts at the right side for didactic clarity.
row_totals = cm_counts_m2.sum(axis=1)
for i, total in enumerate(row_totals):
    ax.text(
        cm_percent_m2.shape[1] + 0.03,
        i + 0.5,
        f"Total={total}",
        transform=ax.transData,
        ha="left",
        va="center",
        fontsize=11,
        color="#374151",
        clip_on=False,
    )

plt.tight_layout()
save_figure(fig, "model2_test_confusion_matrix_row_normalized")
plt.show()

cm_percent_m2_df = pd.DataFrame(cm_percent_m2, index=class_names, columns=class_names)
print("Normalized confusion matrix (%):")
display(cm_percent_m2_df.round(2))

# ROC curve
fpr_m2, tpr_m2, _ = roc_curve(y_true_m2, y_prob_m2)
roc_auc_m2 = auc(fpr_m2, tpr_m2)

fig, ax = plt.subplots(figsize=(7.5, 5.0))
ax.plot(fpr_m2, tpr_m2, lw=2.2, label=f"Model 2 (AUC = {roc_auc_m2:.3f})")
ax.plot([0, 1], [0, 1], lw=1.5, linestyle="--", color="0.5", label="Chance")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("Model 2 ROC Curve", pad=10)
ax.legend(frameon=False, loc="lower right")
ax.grid(True, which="major", alpha=0.25)
plt.tight_layout()
save_figure(fig, "model2_test_roc_curve")
plt.show()

## Model 3: ECG Signals + Demographics (Two-Branch Fusion)
This model combines:
- an **LSTM encoder** for ECG windows
- an **MLP encoder** for demographic features (age, sex, height, weight)

The two embeddings are fused for binary classification (healthy vs non-healthy).

### Dataset & DataLoaders (ECG + Demographics)
This step prepares a dedicated PyTorch dataset and dataloaders for Model 3.
The input uses ECG windows plus patient-level demographic vectors.

In [ ]:
# ------------------------------------------------------------------
# Custom Dataset Class for ECG Windows + Demographics (Model 3)
# ------------------------------------------------------------------

class ECGWindowDemographicsDataset(Dataset):
    """PyTorch Dataset for (ECG window, demographics) pairs."""

    def __init__(self, signals, demographics, labels):
        if len(signals) != len(demographics) or len(signals) != len(labels):
            raise ValueError(
                "signals, demographics, and labels must have the same first dimension"
            )

        self.signals = torch.as_tensor(signals, dtype=torch.float32)
        self.demographics = torch.as_tensor(demographics, dtype=torch.float32)
        self.labels = torch.as_tensor(labels, dtype=torch.float32).unsqueeze(1)  # (n, 1)

    def __len__(self):
        return self.signals.shape[0]

    def __getitem__(self, idx):
        return self.signals[idx], self.demographics[idx], self.labels[idx]

# ------------------------------------------------------------------
# Create datasets + dataloaders for Model 3
# ------------------------------------------------------------------

try:
    batch_size
except NameError:
    batch_size = 32

train_signals_m3 = nn_input_sets["signals_plus_demographics"]["train"]["signals"]
train_demo_m3 = nn_input_sets["signals_plus_demographics"]["train"]["demographics"]
train_labels_m3 = nn_input_sets["signals_plus_demographics"]["train"]["labels"]

test_signals_m3 = nn_input_sets["signals_plus_demographics"]["test"]["signals"]
test_demo_m3 = nn_input_sets["signals_plus_demographics"]["test"]["demographics"]
test_labels_m3 = nn_input_sets["signals_plus_demographics"]["test"]["labels"]

train_dataset_m3 = ECGWindowDemographicsDataset(train_signals_m3, train_demo_m3, train_labels_m3)
test_dataset_m3 = ECGWindowDemographicsDataset(test_signals_m3, test_demo_m3, test_labels_m3)

train_loader_m3 = DataLoader(train_dataset_m3, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader_m3 = DataLoader(test_dataset_m3, batch_size=batch_size, shuffle=False, num_workers=0)

print("=" * 90)
print("DATA LOADERS CREATED - MODEL 3 (ECG + DEMOGRAPHICS)")
print("=" * 90)
print(f"Train samples: {len(train_dataset_m3)} in {len(train_loader_m3)} batches of {batch_size}")
print(f"Test samples:  {len(test_dataset_m3)} in {len(test_loader_m3)} batches of {batch_size}")
print()
print(f"ECG window shape: {train_signals_m3.shape[1:]} (samples, leads)")
print(f"Demographics shape: {train_demo_m3.shape[1:]} (n_demographics_features,)")
print()

batch_sig_m3, batch_demo_m3, batch_lab_m3 = next(iter(train_loader_m3))
print("One batch shapes:")
print(f"  signals:      {tuple(batch_sig_m3.shape)}")
print(f"  demographics: {tuple(batch_demo_m3.shape)}")
print(f"  labels:       {tuple(batch_lab_m3.shape)}")

### Model Definition (ECG + Demographics Fusion Network)
The model has:
- an ECG branch (BiLSTM + projection)
- a demographics branch (small MLP)
- a fusion head for final binary prediction.

In [ ]:
# ------------------------------------------------------------------
# Model 3: ECG + Demographics fusion network
# ------------------------------------------------------------------

class ECGSignalsDemographicsFusionNet(nn.Module):
    """Two-branch network for ECG window classification with demographics."""

    def __init__(
        self,
        n_leads=12,
        n_demo_features=4,
        lstm_hidden=128,
        lstm_layers=2,
        dropout=0.3,
        ecg_embed_dim=64,
        demo_embed_dim=16,
    ):
        super().__init__()

        self.ecg_lstm = nn.LSTM(
            input_size=n_leads,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if lstm_layers > 1 else 0.0,
        )
        self.ecg_proj = nn.Sequential(
            nn.Linear(2 * lstm_hidden, ecg_embed_dim),
            nn.BatchNorm1d(ecg_embed_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        self.demo_mlp = nn.Sequential(
            nn.Linear(n_demo_features, 16),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(16, demo_embed_dim),
            nn.BatchNorm1d(demo_embed_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
        )

        fusion_dim = ecg_embed_dim + demo_embed_dim
        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
            nn.Sigmoid(),
        )

    def forward(self, signals, demographics):
        _, (h_n, _) = self.ecg_lstm(signals)
        ecg_summary = torch.cat([h_n[-2], h_n[-1]], dim=1)  # (batch, 2*lstm_hidden)
        ecg_embed = self.ecg_proj(ecg_summary)
        demo_embed = self.demo_mlp(demographics)
        fused = torch.cat([ecg_embed, demo_embed], dim=1)
        return self.classifier(fused)

n_demo_features = int(nn_input_sets["signals_plus_demographics"]["train"]["demographics"].shape[1])
model_ecg_demo = ECGSignalsDemographicsFusionNet(
    n_leads=12,
    n_demo_features=n_demo_features,
    lstm_hidden=128,
    lstm_layers=2,
    dropout=0.3,
    ecg_embed_dim=64,
    demo_embed_dim=16,
).to(device)

print("=" * 90)
print("MODEL 3: ECG + DEMOGRAPHICS (FUSION)")
print("=" * 90)
print(f"Device: {device}")
print(f"Demographics features: {n_demo_features}")
print()
print(model_ecg_demo)
print()
n_params_m3 = sum(p.numel() for p in model_ecg_demo.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params_m3:,}")

### Training (ECG + Demographics)
Model 3 uses the same training setup as previous models:
- binary cross-entropy loss
- early stopping on validation ROC AUC
- best-checkpoint saving in `../models`.

In [ ]:
# ------------------------------------------------------------------
# Training config + train/validate functions for Model 3
# ------------------------------------------------------------------

learning_rate_m3 = 0.001
num_epochs_m3 = 100
patience_m3 = 15

criterion_m3 = nn.BCELoss()
optimizer_m3 = optim.Adam(model_ecg_demo.parameters(), lr=learning_rate_m3)
scheduler_m3 = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_m3,
    mode="min",
    factor=0.5,
    patience=5,
)

print("=" * 90)
print("MODEL 3 TRAINING CONFIGURATION")
print("=" * 90)
print(f"Loss function: {criterion_m3.__class__.__name__}")
print(f"Optimizer: {optimizer_m3.__class__.__name__}")
print(f"Learning rate: {learning_rate_m3}")
print(f"Max epochs: {num_epochs_m3}")
print(f"Early stopping patience: {patience_m3} epochs")
print()

def _ensure_column_vector_m3(x: torch.Tensor) -> torch.Tensor:
    if x.ndim == 1:
        return x.view(-1, 1)
    if x.ndim == 2 and x.shape[1] == 1:
        return x
    return x.view(-1, 1)

def train_epoch_m3(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    all_probs = []
    all_labels = []

    for batch_signals, batch_demo, batch_labels in train_loader:
        batch_signals = batch_signals.to(device)
        batch_demo = batch_demo.to(device)
        batch_labels = _ensure_column_vector_m3(batch_labels.to(device).float())

        probs = model(batch_signals, batch_demo)
        loss = criterion(probs, batch_labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += float(loss.item())
        all_probs.append(probs.detach().cpu().numpy())
        all_labels.append(batch_labels.detach().cpu().numpy())

    all_probs = np.concatenate(all_probs, axis=0).reshape(-1)
    all_labels = np.concatenate(all_labels, axis=0).reshape(-1).astype(int)
    avg_loss = total_loss / max(1, len(train_loader))
    acc = accuracy_score(all_labels, (all_probs >= 0.5).astype(int))
    return avg_loss, acc

def validate_m3(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for batch_signals, batch_demo, batch_labels in loader:
            batch_signals = batch_signals.to(device)
            batch_demo = batch_demo.to(device)
            batch_labels = _ensure_column_vector_m3(batch_labels.to(device).float())

            probs = model(batch_signals, batch_demo)
            loss = criterion(probs, batch_labels)

            total_loss += float(loss.item())
            all_probs.append(probs.detach().cpu().numpy())
            all_labels.append(batch_labels.detach().cpu().numpy())

    all_probs = np.concatenate(all_probs, axis=0).reshape(-1)
    all_labels = np.concatenate(all_labels, axis=0).reshape(-1).astype(int)
    avg_loss = total_loss / max(1, len(loader))
    acc = accuracy_score(all_labels, (all_probs >= 0.5).astype(int))
    auc_val = roc_auc_score(all_labels, all_probs)
    return avg_loss, acc, auc_val, all_probs, all_labels

#### Training Loop - Model 3 (ECG + Demographics)
The following cell runs training, restores the best validation checkpoint, and saves it.

In [ ]:
# ------------------------------------------------------------------
# Training loop for Model 3 + checkpoint saving
# ------------------------------------------------------------------

model_ecg_demo = ECGSignalsDemographicsFusionNet(
    n_leads=12,
    n_demo_features=n_demo_features,
    lstm_hidden=128,
    lstm_layers=2,
    dropout=0.3,
    ecg_embed_dim=64,
    demo_embed_dim=16,
).to(device)

optimizer_m3 = optim.Adam(model_ecg_demo.parameters(), lr=learning_rate_m3)
scheduler_m3 = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_m3,
    mode="min",
    factor=0.5,
    patience=5,
)

history_m3 = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
    "val_auc": [],
}

best_val_auc_m3 = 0.0
best_val_loss_m3 = float("inf")
best_epoch_m3 = 0
epochs_without_improvement_m3 = 0
best_model_state_m3 = None

print("=" * 90)
print("TRAINING MODEL 3: ECG + DEMOGRAPHICS (FUSION)")
print("=" * 90)
print()

model3_ckpt_name = "model3_ecg_signals_plus_demographics_hiddenstate_best.pt"
model3_ckpt_path = os.path.join(MODELS_DIR, model3_ckpt_name)
num_epochs_to_run_m3 = num_epochs_m3
if os.path.exists(model3_ckpt_path):
    checkpoint_metadata_m3 = load_model_checkpoint(model_ecg_demo, filename=model3_ckpt_name, device=device)
    print(f"Found existing checkpoint '{model3_ckpt_name}'. Skipping training loop.")
    best_model_state_m3 = {k: v.detach().cpu().clone() for k, v in model_ecg_demo.state_dict().items()}
    num_epochs_to_run_m3 = 0
    if isinstance(checkpoint_metadata_m3, dict):
        best_epoch_m3 = int(checkpoint_metadata_m3.get("best_epoch", best_epoch_m3))
        best_val_auc_m3 = float(checkpoint_metadata_m3.get("best_val_auc", best_val_auc_m3))
        best_val_loss_m3 = float(checkpoint_metadata_m3.get("best_val_loss", best_val_loss_m3))

for epoch in range(num_epochs_to_run_m3):
    train_loss, train_acc = train_epoch_m3(
        model_ecg_demo, train_loader_m3, criterion_m3, optimizer_m3, device
    )
    val_loss, val_acc, val_auc, _, _ = validate_m3(
        model_ecg_demo, test_loader_m3, criterion_m3, device
    )

    history_m3["train_loss"].append(train_loss)
    history_m3["train_acc"].append(train_acc)
    history_m3["val_loss"].append(val_loss)
    history_m3["val_acc"].append(val_acc)
    history_m3["val_auc"].append(val_auc)

    scheduler_m3.step(val_loss)

    if val_auc > best_val_auc_m3:
        best_val_auc_m3 = float(val_auc)
        best_val_loss_m3 = float(val_loss)
        best_epoch_m3 = int(epoch + 1)
        epochs_without_improvement_m3 = 0
        best_model_state_m3 = {
            k: v.detach().cpu().clone() for k, v in model_ecg_demo.state_dict().items()
        }
        print(
            f"Epoch {epoch+1:3d} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val AUC: {val_auc:.4f} STAR (BEST)"
        )
    else:
        epochs_without_improvement_m3 += 1
        print(
            f"Epoch {epoch+1:3d} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val AUC: {val_auc:.4f}"
        )

    if epochs_without_improvement_m3 >= patience_m3:
        print(f"\nEarly stopping at epoch {epoch+1}. No improvement for {patience_m3} epochs.")
        break

if best_model_state_m3 is None:
    raise RuntimeError("No best model state found for Model 3.")

model_ecg_demo.load_state_dict(best_model_state_m3)

saved_model3_path = save_model_checkpoint(
    model_ecg_demo,
    filename=model3_ckpt_name,
    metadata={
        "model_name": "ECGSignalsDemographicsFusionNet",
        "best_epoch": int(best_epoch_m3),
        "best_val_auc": float(best_val_auc_m3),
        "best_val_loss": float(best_val_loss_m3),
        "sampling_rate_hz": int(sampling_rate_to_plot),
        "window_sec": float(WINDOW_SEC),
        "step_sec": float(STEP_SEC),
        "n_demographics_features": int(n_demo_features),
    },
)

print()
print("=" * 90)
print("MODEL 3 TRAINING COMPLETE")
print("=" * 90)
print(f"Best epoch: {best_epoch_m3}")
print(f"Best Validation Loss: {best_val_loss_m3:.4f}")
print(f"Best Validation AUC: {best_val_auc_m3:.4f}")
print(f"Total epochs trained: {len(history_m3['train_loss'])}")
print(f"Saved best model to: {saved_model3_path}")

### Training History & Performance Visualization (Model 3)

In [ ]:
# Plot training history for Model 3
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Loss curve
axes[0].plot(history_m3['train_loss'], label='Train Loss', linewidth=2, color='steelblue')
axes[0].plot(history_m3['val_loss'], label='Val Loss', linewidth=2, color='coral')
axes[0].set_xlabel('Epoch', fontsize=11, fontweight='bold')
axes[0].set_ylabel('BCE Loss', fontsize=11, fontweight='bold')
axes[0].set_title('Training & Validation Loss', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

# Accuracy curve
axes[1].plot(history_m3['train_acc'], label='Train Accuracy', linewidth=2, color='steelblue')
axes[1].plot(history_m3['val_acc'], label='Val Accuracy', linewidth=2, color='coral')
axes[1].set_xlabel('Epoch', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Accuracy', fontsize=11, fontweight='bold')
axes[1].set_title('Training & Validation Accuracy', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)

# AUC curve
axes[2].plot(history_m3['val_auc'], label='Val ROC-AUC', linewidth=2, color='mediumseagreen')
axes[2].set_xlabel('Epoch', fontsize=11, fontweight='bold')
axes[2].set_ylabel('ROC-AUC', fontsize=11, fontweight='bold')
axes[2].set_title('Validation ROC-AUC', fontsize=12, fontweight='bold')
axes[2].legend(fontsize=10)
axes[2].grid(alpha=0.3)
axes[2].set_ylim([0, 1.05])

plt.tight_layout()
save_figure(fig, "model3_training_history")
plt.show()

print("Model 3 training history plots saved.")
print()

### Final Model Evaluation on Test Set (Model 3)
This section loads the best Model 3 checkpoint, evaluates on test data, and plots confusion matrix plus ROC curve.

In [ ]:
# ------------------------------------------------------------------
# Final Model 3 evaluation on test set (load best checkpoint)
# ------------------------------------------------------------------

model3_ckpt_name = "model3_ecg_signals_plus_demographics_hiddenstate_best.pt"
model_ecg_demo_eval = ECGSignalsDemographicsFusionNet(
    n_leads=12,
    n_demo_features=n_demo_features,
    lstm_hidden=128,
    lstm_layers=2,
    dropout=0.3,
    ecg_embed_dim=64,
    demo_embed_dim=16,
).to(device)

_ = load_model_checkpoint(model_ecg_demo_eval, filename=model3_ckpt_name, device=device)
model_ecg_demo_eval.eval()

y_true_m3 = []
y_prob_m3 = []
y_pred_m3 = []

with torch.no_grad():
    for signals, demographics, labels in test_loader_m3:
        signals = signals.to(device)
        demographics = demographics.to(device)
        labels = labels.to(device).float().view(-1, 1)

        probs = model_ecg_demo_eval(signals, demographics)
        preds = (probs >= 0.5).float()

        y_true_m3.append(labels.detach().cpu().numpy())
        y_prob_m3.append(probs.detach().cpu().numpy())
        y_pred_m3.append(preds.detach().cpu().numpy())

y_true_m3 = np.concatenate(y_true_m3, axis=0).reshape(-1).astype(int)
y_pred_m3 = np.concatenate(y_pred_m3, axis=0).reshape(-1).astype(int)
y_prob_m3 = np.concatenate(y_prob_m3, axis=0).reshape(-1).astype(float)

acc_m3 = accuracy_score(y_true_m3, y_pred_m3)
auc_m3 = roc_auc_score(y_true_m3, y_prob_m3)
print("=" * 90)
print("MODEL 3: FINAL EVALUATION (TEST SET)")
print("=" * 90)
print(f"Accuracy: {acc_m3:.4f}")
print(f"ROC AUC:  {auc_m3:.4f}")
print()
print("Classification report:")
print(classification_report(y_true_m3, y_pred_m3, digits=4))

cm_counts_m3 = confusion_matrix(y_true_m3, y_pred_m3, labels=[0, 1])
cm_percent_m3 = confusion_matrix(y_true_m3, y_pred_m3, labels=[0, 1], normalize="true") * 100.0
class_names = ["Healthy (0)", "Not Healthy (1)"]

fig, ax = plt.subplots(figsize=(8.8, 7.0))
with sns.axes_style("white"):
    heat = sns.heatmap(
        cm_percent_m3,
        annot=False,
        fmt=".1f",
        cmap="Blues",
        vmin=0,
        vmax=100,
        cbar=True,
        cbar_kws={
            "label": "Row-normalized percentage [%]",
            "shrink": 0.65,
            "pad": 0.15,
        },
        xticklabels=class_names,
        yticklabels=class_names,
        linewidths=1.6,
        linecolor="white",
        square=True,
        ax=ax,
    )

ax.set_xlabel("Predicted label", fontweight="bold", labelpad=10)
ax.set_ylabel("True label", fontweight="bold", labelpad=10)
ax.set_title("Normalized Confusion Matrix (%) - Model 3", fontweight="bold", pad=14)
ax.set_xticklabels(class_names, rotation=0, ha="center")
ax.set_yticklabels(class_names, rotation=0, va="center")

cbar = heat.collections[0].colorbar
cbar.ax.tick_params(labelsize=11)
cbar.set_label("Row-normalized percentage [%]", fontsize=12)

threshold = 60.0
for i in range(cm_percent_m3.shape[0]):
    for j in range(cm_percent_m3.shape[1]):
        pct = cm_percent_m3[i, j]
        cnt = cm_counts_m3[i, j]
        text_color = "white" if pct >= threshold else "#1f2937"
        ax.text(
            j + 0.5,
            i + 0.5,
            f"{pct:.1f}%\n(n={cnt})",
            ha="center",
            va="center",
            fontsize=12,
            fontweight="bold",
            color=text_color,
        )

row_totals = cm_counts_m3.sum(axis=1)
for i, total in enumerate(row_totals):
    ax.text(
        cm_percent_m3.shape[1] + 0.03,
        i + 0.5,
        f"Total={total}",
        transform=ax.transData,
        ha="left",
        va="center",
        fontsize=11,
        color="#374151",
        clip_on=False,
    )

plt.tight_layout()
save_figure(fig, "model3_test_confusion_matrix_row_normalized")
plt.show()

cm_percent_m3_df = pd.DataFrame(cm_percent_m3, index=class_names, columns=class_names)
print("Normalized confusion matrix (%):")
display(cm_percent_m3_df.round(2))

fpr_m3, tpr_m3, _ = roc_curve(y_true_m3, y_prob_m3)
roc_auc_m3 = auc(fpr_m3, tpr_m3)

fig, ax = plt.subplots(figsize=(7.5, 5.0))
ax.plot(fpr_m3, tpr_m3, lw=2.2, label=f"Model 3 (AUC = {roc_auc_m3:.3f})")
ax.plot([0, 1], [0, 1], lw=1.5, linestyle="--", color="0.5", label="Chance")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("Model 3 ROC Curve", pad=10)
ax.legend(frameon=False, loc="lower right")
ax.grid(True, which="major", alpha=0.25)
plt.tight_layout()
save_figure(fig, "model3_test_roc_curve")
plt.show()

## Final Comparison: Model 1 vs Model 2 vs Model 3
This section compares the three trained models on the same test set:
- **Model 1**: ECG only
- **Model 2**: ECG + HRV
- **Model 3**: ECG + demographics

We compare both:
1. row-normalized confusion matrices (%)
2. ROC curves in a single plot.

In [ ]:
# ------------------------------------------------------------------
# Compare Model 1 / Model 2 / Model 3 on confusion matrices + ROC
# ------------------------------------------------------------------

if "ECGSignalsOnlyLSTM" not in globals():
    raise RuntimeError(
        "ECGSignalsOnlyLSTM is not defined in the current kernel. "
        "Run the Model 1 architecture cell before this comparison cell."
    )
if "ECGSignalsHRVFusionNet" not in globals():
    raise RuntimeError(
        "ECGSignalsHRVFusionNet is not defined in the current kernel. "
        "Run the Model 2 architecture cell before this comparison cell."
    )
if "ECGSignalsDemographicsFusionNet" not in globals():
    raise RuntimeError(
        "ECGSignalsDemographicsFusionNet is not defined in the current kernel. "
        "Run the Model 3 architecture cell before this comparison cell."
    )

# Build dedicated test loaders (self-contained comparison cell).
test_signals_m1 = nn_input_sets["signals_only"]["test"]["signals"]
test_labels_m1 = nn_input_sets["signals_only"]["test"]["labels"]
test_tensor_m1 = TensorDataset(
    torch.as_tensor(test_signals_m1, dtype=torch.float32),
    torch.as_tensor(test_labels_m1, dtype=torch.float32).view(-1, 1),
)
test_loader_cmp_m1 = DataLoader(test_tensor_m1, batch_size=batch_size, shuffle=False, num_workers=0)

test_signals_m2 = nn_input_sets["signals_plus_hrv"]["test"]["signals"]
test_hrv_m2 = nn_input_sets["signals_plus_hrv"]["test"]["hrv"]
test_labels_m2 = nn_input_sets["signals_plus_hrv"]["test"]["labels"]
test_tensor_m2 = TensorDataset(
    torch.as_tensor(test_signals_m2, dtype=torch.float32),
    torch.as_tensor(test_hrv_m2, dtype=torch.float32),
    torch.as_tensor(test_labels_m2, dtype=torch.float32).view(-1, 1),
)
test_loader_cmp_m2 = DataLoader(test_tensor_m2, batch_size=batch_size, shuffle=False, num_workers=0)

test_signals_m3 = nn_input_sets["signals_plus_demographics"]["test"]["signals"]
test_demo_m3 = nn_input_sets["signals_plus_demographics"]["test"]["demographics"]
test_labels_m3 = nn_input_sets["signals_plus_demographics"]["test"]["labels"]
test_tensor_m3 = TensorDataset(
    torch.as_tensor(test_signals_m3, dtype=torch.float32),
    torch.as_tensor(test_demo_m3, dtype=torch.float32),
    torch.as_tensor(test_labels_m3, dtype=torch.float32).view(-1, 1),
)
test_loader_cmp_m3 = DataLoader(test_tensor_m3, batch_size=batch_size, shuffle=False, num_workers=0)


In [ ]:
# Instantiate and load best checkpoints.
model_m1_cmp = ECGSignalsOnlyLSTM(
    n_leads=12,
    lstm_hidden=128,
    lstm_layers=2,
    dropout=0.3,
).to(device)
_ = load_model_checkpoint(model_m1_cmp, "model1_ecg_signals_only_hiddenstate_best.pt", device)
model_m1_cmp.eval()

n_hrv_features_cmp = int(nn_input_sets["signals_plus_hrv"]["train"]["hrv"].shape[1])
model_m2_cmp = ECGSignalsHRVFusionNet(
    n_leads=12,
    n_hrv_features=n_hrv_features_cmp,
    lstm_hidden=128,
    lstm_layers=2,
    dropout=0.3,
    ecg_embed_dim=64,
    hrv_embed_dim=16,
).to(device)
_ = load_model_checkpoint(model_m2_cmp, "model2_ecg_signals_plus_hrv_hiddenstate_best.pt", device)
model_m2_cmp.eval()

n_demo_features_cmp = int(nn_input_sets["signals_plus_demographics"]["train"]["demographics"].shape[1])
model_m3_cmp = ECGSignalsDemographicsFusionNet(
    n_leads=12,
    n_demo_features=n_demo_features_cmp,
    lstm_hidden=128,
    lstm_layers=2,
    dropout=0.3,
    ecg_embed_dim=64,
    demo_embed_dim=16,
).to(device)
_ = load_model_checkpoint(model_m3_cmp, "model3_ecg_signals_plus_demographics_hiddenstate_best.pt", device)
model_m3_cmp.eval()

def predict_m1(model, loader, device):
    y_true, y_prob = [], []
    with torch.no_grad():
        for signals, labels in loader:
            signals = signals.to(device)
            labels = labels.to(device).float().view(-1, 1)
            probs = model(signals)
            y_true.append(labels.detach().cpu().numpy())
            y_prob.append(probs.detach().cpu().numpy())
    y_true = np.concatenate(y_true, axis=0).reshape(-1).astype(int)
    y_prob = np.concatenate(y_prob, axis=0).reshape(-1).astype(float)
    y_pred = (y_prob >= 0.5).astype(int)
    return y_true, y_prob, y_pred

def predict_m2(model, loader, device):
    y_true, y_prob = [], []
    with torch.no_grad():
        for signals, hrv, labels in loader:
            signals = signals.to(device)
            hrv = hrv.to(device)
            labels = labels.to(device).float().view(-1, 1)
            probs = model(signals, hrv)
            y_true.append(labels.detach().cpu().numpy())
            y_prob.append(probs.detach().cpu().numpy())
    y_true = np.concatenate(y_true, axis=0).reshape(-1).astype(int)
    y_prob = np.concatenate(y_prob, axis=0).reshape(-1).astype(float)
    y_pred = (y_prob >= 0.5).astype(int)
    return y_true, y_prob, y_pred

def predict_m3(model, loader, device):
    y_true, y_prob = [], []
    with torch.no_grad():
        for signals, demographics, labels in loader:
            signals = signals.to(device)
            demographics = demographics.to(device)
            labels = labels.to(device).float().view(-1, 1)
            probs = model(signals, demographics)
            y_true.append(labels.detach().cpu().numpy())
            y_prob.append(probs.detach().cpu().numpy())
    y_true = np.concatenate(y_true, axis=0).reshape(-1).astype(int)
    y_prob = np.concatenate(y_prob, axis=0).reshape(-1).astype(float)
    y_pred = (y_prob >= 0.5).astype(int)
    return y_true, y_prob, y_pred

y_true_m1_cmp, y_prob_m1_cmp, y_pred_m1_cmp = predict_m1(model_m1_cmp, test_loader_cmp_m1, device)
y_true_m2_cmp, y_prob_m2_cmp, y_pred_m2_cmp = predict_m2(model_m2_cmp, test_loader_cmp_m2, device)
y_true_m3_cmp, y_prob_m3_cmp, y_pred_m3_cmp = predict_m3(model_m3_cmp, test_loader_cmp_m3, device)

if not (np.array_equal(y_true_m1_cmp, y_true_m2_cmp) and np.array_equal(y_true_m1_cmp, y_true_m3_cmp)):
    raise RuntimeError("Test labels mismatch across model inputs; cannot compare fairly.")

y_true_cmp = y_true_m1_cmp
class_names_cmp = ["Healthy (0)", "Not Healthy (1)"]

model_results = [
    {"name": "Model 1 (ECG)", "y_prob": y_prob_m1_cmp, "y_pred": y_pred_m1_cmp},
    {"name": "Model 2 (ECG + HRV)", "y_prob": y_prob_m2_cmp, "y_pred": y_pred_m2_cmp},
    {"name": "Model 3 (ECG + Demo)", "y_prob": y_prob_m3_cmp, "y_pred": y_pred_m3_cmp},
]


In [ ]:
# 1) Confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5.8), constrained_layout=True)
for ax, result in zip(axes, model_results):
    cm_counts = confusion_matrix(y_true_cmp, result["y_pred"], labels=[0, 1])
    cm_percent = confusion_matrix(y_true_cmp, result["y_pred"], labels=[0, 1], normalize="true") * 100.0

    sns.heatmap(
        cm_percent,
        annot=False,
        cmap="Blues",
        vmin=0,
        vmax=100,
        cbar=True,
        cbar_kws={"label": "%", "shrink": 0.70, "pad": 0.06},
        xticklabels=class_names_cmp,
        yticklabels=class_names_cmp,
        linewidths=1.2,
        linecolor="white",
        square=True,
        ax=ax,
    )

    for i in range(cm_percent.shape[0]):
        for j in range(cm_percent.shape[1]):
            pct = cm_percent[i, j]
            cnt = cm_counts[i, j]
            txt_color = "white" if pct >= 60 else "#1f2937"
            ax.text(j + 0.5, i + 0.5, f"{pct:.1f}%\n(n={cnt})", ha="center", va="center", fontsize=10, fontweight="bold", color=txt_color)

    ax.set_title(result["name"], fontweight="bold", pad=10)
    ax.set_xlabel("Predicted", fontweight="bold")
    ax.set_ylabel("True", fontweight="bold")
    ax.set_xticklabels(class_names_cmp, rotation=20, ha="center")
    ax.set_yticklabels(class_names_cmp, rotation=0, va="center")

plt.suptitle("Confusion Matrix Comparison (Row-normalized %)", fontsize=14, fontweight="bold", y=1.02)
save_figure(fig, "models_comparison_confusion_matrices_row_normalized")
plt.show()

# 2) ROC curves
fig, ax = plt.subplots(figsize=(8.2, 6.0))
for result in model_results:
    fpr, tpr, _ = roc_curve(y_true_cmp, result["y_prob"])
    auc_score = auc(fpr, tpr)
    acc_score = accuracy_score(y_true_cmp, result["y_pred"])
    ax.plot(fpr, tpr, lw=2.2, label=f"{result['name']} | AUC={auc_score:.3f} | Acc={acc_score:.3f}")

ax.plot([0, 1], [0, 1], linestyle="--", color="0.5", lw=1.4, label="Chance")
ax.set_xlim([-0.01, 1.0])
ax.set_ylim([0.0, 1.02])
ax.set_xlabel("False Positive Rate", fontweight="bold")
ax.set_ylabel("True Positive Rate", fontweight="bold")
ax.set_title("ROC Curve Comparison - Models 1/2/3", fontweight="bold", pad=10)
ax.grid(alpha=0.25)
ax.legend(frameon=False, loc="lower right")
plt.tight_layout()
save_figure(fig, "models_comparison_roc_curves")
plt.show()

summary_rows = []
for result in model_results:
    summary_rows.append({
        "Model": result["name"],
        "Accuracy": round(accuracy_score(y_true_cmp, result["y_pred"]), 4),
        "ROC_AUC": round(roc_auc_score(y_true_cmp, result["y_prob"]), 4),
    })
summary_df = pd.DataFrame(summary_rows).sort_values(by="ROC_AUC", ascending=False).reset_index(drop=True)
print("Summary (sorted by ROC AUC):")
display(summary_df)